# Chess — ThompsonZero with a STATE-VALUE network (child-batch expansion)

A rework of the chess ThompsonZero notebook (`claude/thompson_z`) around one
architectural change: **the network predicts the value of the current
position, not a value per action.**

### The change

The per-action design predicted `(p_win, p_draw, p_loss, confidence)` for all
4674 action slots from one forward pass at the parent position — two flat
`Linear(512, 4674·k)` heads holding ~9.6M of the net's 12.7M parameters,
asked to rank moves whose *resulting positions the network never sees*.

This notebook's `ThompsonValueChessNet` predicts **one Dirichlet over the
current state's (win, draw, loss)** plus one confidence — 4 outputs. When the
search expands a node, it applies every legal move and **batch-evaluates all
resulting child positions in a single forward pass**. Each edge's belief is
the child state's predicted Dirichlet with the win/loss columns swapped —
an *exact* perspective flip (chess strictly alternates movers), not an
approximation. Everything downstream — the value-grid PMFs, P(best) mixture
propagation, Thompson selection, MCTS-Solver, innovation calibration, tree
reuse, z-mix, restart curriculum — is unchanged in mechanism.

### Why this should be better

1. **Smaller net, all of it working on the value.** 64ch × 6 blocks ≈ 0.6M
   parameters (vs 12.7M). No capacity is spent addressing head slots; the
   only learning problem is "how good is this position", which is also the
   only quantity the search ever consumes.
2. **Higher-fidelity edge signal.** Every edge belief is grounded in a real
   evaluation of the position the move actually produces. A per-action head
   has to *infer* move quality from the parent position; a state net gets to
   *look*. The root's edges each carry an independent real evaluation from
   sim 0.
3. **Free, earlier solver proofs.** Expansion applies every move, so every
   terminal child (mate, stalemate, repetition/50-move/material draw) is
   detected and enters the tree as an exact MCTS-Solver proof — a mate-in-1
   anywhere below an expanded node is proven at construction, before any
   simulation finds it. The search-free evaluation player (`value1` track)
   also natively plays mates and avoids stalemates for the same reason.
4. **GPU-shaped batches.** Each expansion ships ~35 rows instead of 1, so
   worker requests are ~1100 rows at the defaults and the inference server's
   batches are an order of magnitude larger — exactly the regime the
   DirectML batching architecture was built for. The response wire format is
   16 bytes/row with **no gathering machinery at all** (the per-action
   notebook needed probed on-device `index_select` to avoid 74.8 KB dense
   rows; here the dense output *is* 16 bytes).

### The cost accounting (why the sim budgets changed)

Expanding a node costs ~35 net evaluations instead of 1, but each row is ~7×
cheaper (the conv body dominated forward FLOPs and shrank 128×10 → 64×6), so
GPU work **per simulation** is ~5× the old design's. The budgets are cut ~4×
(`FAST 250→64`, `FULL 1000→256`), leaving GPU work per move roughly equal.
Information per move is far higher: a 64-sim tree contains ~64 expanded nodes
≈ **2,200 evaluated positions** (the old 250-sim tree evaluated 250), every
edge grounded one ply deeper than before. The new CPU cost is the ~35
`clone()+apply()+observation_tensor()` calls per expansion — workers are
leaner (`GAMES_PER_WORKER 32→16`, `WORKER_WAVE 8→4`) since each request is
already large.

### Training target

One `(obs fp16, counts fp32 (3,))` pair per position: the root's posterior
mixture belief moment-matched back to (p_win, p_draw, p_loss) × concentration
(from `Var[v]`, capped at `CONF_CAP`), or the exact spike for solver-proven
roots. Losses are the same Dirichlet-Multinomial evidence NLL (default) / KL
(A/B) — now **dense and trivially small** (`(B,3)` + `(B,)`), so the DirectML
lgamma split-backward needs no gather and the sparse-target machinery is gone.

Notes on the evidence-only principle: the per-action notebook excluded
unexpanded edges from targets so the loss never scored the net against its
own echoed prior. Here the question resolves structurally: expansion
evaluates **all** children at once, so "unexpanded edges of an expanded
node" don't exist — every edge in a target's mixture is grounded in a real
child evaluation (or a proof). The z-mix also simplifies: the game outcome
is one real multinomial observation of *every* recorded position's outcome
under the policy (no played-action caveat needed — a state target has no
siblings), still γ-ramped from the game's end and skipped for cutoffs and
proven targets.

### Carried over unchanged from `claude/thompson_z`

Mixture propagation (Tesauro P(best) weights), the value-grid Dirichlet
decomposition (v = p_win − p_loss with the (1−d) draw shrinkage baked in),
MCTS-Solver with WIN > DRAW > LOSS precedence, innovation-based confidence
calibration (empirical-Bayes λ, no caps), tree reuse across moves
(`_descend`), solver-labeled auxiliary samples, γ-ramped Z_MIX, the
backward-restart curriculum + random opponent pool, opponent-pool diversity,
the multiprocess self-play pool with a central GPU inference server, all
DirectML workarounds (`LerpFreeAdamW`, elementwise GroupNorm/softplus, split
backward, model lock, CPU eval replicas, main-thread autograd warm-up),
running-Elo benchmark tournaments, checkpoint resume with benchmark dedup,
and the self-play/buffer diagnostics.

### Network architecture

```
Input (20, 8, 8) — chess's native observation planes
                 →  Stem (Conv 3×3, GroupNorm, ReLU)
                 →  6 × ResBlock(64) (Conv-GN-ReLU-Conv-GN + SE) + skip
                 →  Head: 1×1 conv (8 ch) → flatten (512)
                       → Linear(3) → softmax → (p_win, p_draw, p_loss)
                       → Linear(1) → softplus → confidence α₀ ∈ [CONF_MIN, CONF_MAX]
```

### Things to watch in a real run

- **Worker CPU is the likely bottleneck** (35 clone+obs calls per expansion).
  If the GPU idles, raise `SELFPLAY_WORKERS` / `GAMES_PER_WORKER` before
  touching sim budgets.
- **Transpositions cost more here**: two moves reaching the same position
  evaluate it twice (once per parent). A per-expansion memo table would be a
  natural follow-up if profiling shows it matters.
- The `value1` eval track (search-free, one-ply) replaces the old raw-policy
  track and is strictly stronger than a raw forward (it sees terminals);
  compare tracks within this notebook, not across notebooks.
- **Calibration's ground truth now flows through proofs.** A one-ply
  innovation here compares the same net at neighboring inputs (direct child
  eval vs mixture of grandchild evals), so in quiet positions a smooth net
  looks self-consistent regardless of whether its confidence is deserved —
  λ can drift below 1 early. The refutation channel is solver proofs
  (terminal children score held beliefs against the actual ±1/0), which
  this design produces far more of than the per-action one (every terminal
  one ply below any expanded node), amplified further by the restart
  curriculum. Watch `lam=` alongside `aux=`: λ stuck below 1 while quick
  evals stagnate AND `aux` stays near zero would mean the net is
  sharpening on self-consistency without ever meeting truth — raise
  `RESTART_PROB` / `RANDOM_POOL_FRAC` in that case.

In [ ]:
%pip install open_spiel -q
# AMD/Intel GPU acceleration via DirectML (Windows / WSL2). Needs Python
# 3.11 or 3.12 (torch-directml has no 3.13 wheel) and pins torch==2.3.1.
# Harmless if you're on CUDA/CPU — the device code below falls back.
%pip install torch-directml -q

In [ ]:
import pyspiel

# Chess is already implemented natively in OpenSpiel (C++, games/chess/) — no
# custom engine needed here, unlike the Boop notebooks. It provides exact
# detection of every real chess draw (insufficient material, threefold
# repetition, stalemate, the 50-move rule) as well as checkmate, all inside
# is_terminal()/returns(); utilities are exactly WinUtility=+1, DrawUtility=0,
# LossUtility=-1, matching this notebook's v = p_win - p_loss value scale
# with no rescaling needed.
#
# Board-plane convention (see chess.cc ObservationTensor): the 20 planes are
# ABSOLUTE, not player-relative — White/Black pieces occupy fixed planes
# regardless of who is asking, with a side-to-play plane telling the network
# whose turn it is (unlike Boop's mover-relative flat observation). This is
# also why there is no board-symmetry augmentation available for chess (a
# left-right mirror is not a symmetry of the starting position: King/Queen
# occupy distinct files) — see the intro cell for a color-flip 180° rotation
# that WOULD be a valid symmetry, left unimplemented for now.
GAME = pyspiel.load_game('chess')
_NUM_ACTIONS = GAME.num_distinct_actions()
_OBS_SHAPE   = tuple(GAME.observation_tensor_shape())    # (20, 8, 8)

print('Game:', GAME.get_type().long_name)
print('Actions:', _NUM_ACTIONS)
print('Observation shape:', _OBS_SHAPE)
print('Utility (min/max):', GAME.min_utility(), GAME.max_utility())

_state = GAME.new_initial_state()
print()
print(_state)
print('Legal actions from the start position:', len(_state.legal_actions()))
del _state

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import random

# ── Device selection ──────────────────────────────────────────────────────────
# Same policy as the Boop/chess Thompson notebooks: the GPU only wins with BIG
# batches, so self-play NN requests are funneled through one batching server
# and training runs large-batch steps there; small-batch paths (eval, arena)
# use CPU replicas. The state-value design below makes the batches BIGGER
# still (every node expansion ships ~35 child positions at once), which is
# exactly the regime DirectML wants.
DEVICE_PREFERENCE = 'directml' # 'cpu' | 'directml' | 'cuda' | 'auto'

def _pick_device(pref):
    if pref in ('directml', 'auto'):
        try:
            import torch_directml
            try:    _name = torch_directml.device_name(0)
            except Exception: _name = 'DirectML GPU'
            print(f'Using DirectML: {_name}')
            return torch_directml.device(), 'directml'
        except Exception:
            if pref == 'directml':
                print('DirectML requested but unavailable — falling back.')
    if pref in ('cuda', 'auto') and torch.cuda.is_available():
        return torch.device('cuda'), 'cuda'
    return torch.device('cpu'), 'cpu'

DEVICE, _BACKEND = _pick_device(DEVICE_PREFERENCE)


# ── Outcome convention (used EVERYWHERE in this notebook) ─────────────────────
# The network predicts ONE Dirichlet over the CURRENT STATE's 3-way outcome
# (win, draw, loss) FROM THE PERSPECTIVE OF THE PLAYER TO MOVE — not a
# per-action head. Per-EDGE beliefs (what the tree machinery consumes) are
# obtained by evaluating each CHILD state and flipping perspective: chess
# strictly alternates movers, so the child's (win, draw, loss) seen by the
# child's mover is exactly the parent mover's (loss, draw, win) for the action
# that led there — a pure column swap, no approximation. The scalar value used
# for selection/comparison is v = p_win − p_loss ∈ [−1, 1], chess's own
# utility scale (WinUtility=+1, DrawUtility=0, LossUtility=−1).
_WIN, _DRAW, _LOSS = 0, 1, 2                       # outcome columns (win, draw, loss)
_TERM_DRAW = _DRAW                                  # alias: solver/terminal draw marker
# MCTS-Solver: a proven outcome for the mover flips one ply up (win<->loss,
# draw stays), because the parent's mover is the opponent. Indexed by outcome.
_FLIP_TERM = np.array([_LOSS, _DRAW, _WIN], dtype=np.int8)

ALPHA_FLOOR = 0.05    # per-component Dirichlet floor: keeps discretization + NLL finite
CONF_MIN    = 0.5     # smallest predictable prior strength (pseudo-observations)
CONF_MAX    = 100.0   # cap: stops confidence from running away across generations


# ── Input helpers ─────────────────────────────────────────────────────────────
# Chess's observation tensor is ALREADY a dense (20, 8, 8) plane stack (12
# piece planes + 1 empty plane + repetition-count + side-to-play +
# irreversible-move-counter + 4 castling-rights planes) — no reshape needed.
# It is NOT player-relative (White and Black pieces occupy fixed planes
# regardless of who is asking; a side-to-play plane tells the network whose
# turn it is), which is also why there is no useful board-symmetry
# augmentation for chess — see the intro cell.

def state_to_tensor(state, device):
    """Single game state → (1, 20, 8, 8) float tensor."""
    obs = np.array(state.observation_tensor(state.current_player()), dtype=np.float32)
    x   = obs.reshape(1, *_OBS_SHAPE)
    return torch.from_numpy(x).to(device)


def batch_to_tensor(obs_list, device):
    """List of flat observations → (B, 20, 8, 8) float tensor. Accepts python
    lists or numpy arrays of any float dtype (replay samples and wire traffic
    are fp16 — see the tree-ops cell) and converts in ONE pass."""
    obs = np.asarray(obs_list, dtype=np.float32)
    x   = obs.reshape(-1, *_OBS_SHAPE)
    return torch.from_numpy(x).to(device)


# ── Network modules (identical to the Boop/chess Thompson notebooks) ──────────

class _GroupNorm(nn.Module):
    """GroupNorm from elementwise ops — DirectML-safe (the fused kernel's
    backward is broken there) and keeps NO running stats (train == eval)."""
    def __init__(self, num_groups, num_channels, eps=1e-5):
        super().__init__()
        self.num_groups = num_groups
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))

    def forward(self, x):
        n, c = x.shape[0], x.shape[1]
        xg = x.reshape(n, self.num_groups, -1)
        mean = xg.mean(dim=2, keepdim=True)
        var = (xg - mean).pow(2).mean(dim=2, keepdim=True)
        xg = (xg - mean) / torch.sqrt(var + self.eps)
        x = xg.reshape(x.shape)
        return x * self.weight.view(1, c, 1, 1) + self.bias.view(1, c, 1, 1)


def _norm(channels):
    groups = min(8, channels)
    while channels % groups != 0:
        groups -= 1
    return _GroupNorm(groups, channels)


def _softplus(x):
    """Numerically-stable softplus from elementary ops only. torch's fused
    aten::softplus has no DirectML kernel; relu/abs/exp/log/add all do.
    Identical to F.softplus: log(1+e^x) = max(x,0) + log(1 + e^-|x|)."""
    return torch.relu(x) + torch.log(1.0 + torch.exp(-torch.abs(x)))


class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (KataGo-style)."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels * 2),
        )

    def forward(self, x):
        s = x.mean(dim=(2, 3))
        scale, bias = self.fc(s).chunk(2, dim=1)
        scale = torch.sigmoid(scale)
        return (x * scale[:, :, None, None]
                  + bias[:, :, None, None])


class ResBlock(nn.Module):
    """Residual block: Conv-GN-ReLU-Conv-GN + SE attention + skip."""
    def __init__(self, channels, use_se=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
        )
        self.se  = SEBlock(channels) if use_se else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.se(self.net(x)) + x)


class ThompsonValueChessNet(nn.Module):
    """
    ThompsonZero STATE-VALUE network for Chess: one 3-way outcome distribution
    plus one confidence for the CURRENT POSITION (mover's perspective) —
    replacing the per-action heads of the previous chess Thompson notebook
    (dist_out Linear(512, 4674·3) + conf_out Linear(512, 4674), ~9.6M of that
    net's 12.7M parameters). Per-action beliefs are produced instead by
    evaluating each legal move's RESULTING position (batched — see the
    tree-ops cell), which grounds every edge in a real evaluation rather than
    a head-slot lookup, and turns each node expansion into one large
    GPU-friendly batch.

    Input  : (B, 20, 8, 8) — chess's native observation planes
    Body   : Conv stem → N × ResBlock(channels, SE)
    Head   : 1×1 conv (H ch) → flatten →
               dist_out Linear(H·64, 3) → softmax → (p_win, p_draw, p_loss)
               conf_out Linear(H·64, 1) → softplus → α₀ ∈ [CONF_MIN, CONF_MAX]

    forward(x) → (probs (B, 3), conf (B,)).
    The predicted belief for the state is Dirichlet(conf·probs) — the
    (win, draw, loss) pseudo-counts (conf·p_win, conf·p_draw, conf·p_loss).

    Every parameter now works on the value estimate: no capacity is spent
    ranking 4674 action slots, and the (small) output means the whole
    forward-to-consumer path is bandwidth-trivial (4 floats/row vs 18.7K).
    """
    _HEAD_CH = 8

    def __init__(self, channels=64, num_blocks=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(_OBS_SHAPE[0], channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
        )
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.Conv2d(channels, self._HEAD_CH, 1, bias=False),
            _norm(self._HEAD_CH),
            nn.ReLU(inplace=True),
            nn.Flatten(),
        )
        flat = self._HEAD_CH * _OBS_SHAPE[1] * _OBS_SHAPE[2]
        self.dist_out = nn.Linear(flat, 3)
        self.conf_out = nn.Linear(flat, 1)
        # Start near-uniform (p_win=p_draw=p_loss=1/3) with a weak prior
        # (α₀ ≈ 2.4): an untrained net should claim LOW confidence so search
        # observations dominate the posterior from the very first generation.
        nn.init.constant_(self.conf_out.bias, 1.4)   # CONF_MIN + softplus(1.4) ≈ 2.4

    def forward(self, x):
        h     = self.head(self.body(self.stem(x)))
        probs = F.softmax(self.dist_out(h), dim=-1)
        conf  = torch.clamp(CONF_MIN + _softplus(self.conf_out(h)).squeeze(-1),
                            max=CONF_MAX)
        return probs, conf


print(f'Device: {DEVICE}')
_demo = ThompsonValueChessNet()
print(f'ThompsonValueChessNet params: {sum(p.numel() for p in _demo.parameters()):,}')
del _demo

In [ ]:
# ── Training-sample format ─────────────────────────────────────────────────────
# A sample is (obs fp16 (1280,), counts fp32 (3,)): the position and its
# posterior (win, draw, loss) pseudo-counts from the MOVER's perspective.
# That's it — no per-action targets, no sparse index bookkeeping. The
# state-value design moved all per-action structure out of the network and
# into the SEARCH (each edge's belief comes from actually evaluating the
# child position), so the training target collapses to one 3-vector per
# position: ~2.6 KB/sample all-in (the fp16 observation dominates).

def augment_sample(obs_flat, counts, solved=False):
    """No-op: returns the single unaugmented sample as (obs, counts, solved),
    wrapped in a list for a call-site identical to the earlier notebooks
    (`replay_buffer.extend(augment_sample(…))`) — chess has no usable board
    symmetry (see the intro cell). `solved` tags a solver-labeled
    (MCTS-proven, exact ground-truth) position — a diagnostic flag only; the
    training loss ignores it."""
    return [(np.asarray(obs_flat, dtype=np.float16),
             np.asarray(counts,   dtype=np.float32),
             bool(solved))]


# ── Bayesian-MCTS tree: MIXTURE-propagated DIRICHLET value distributions ──────
# Identical belief machinery to the per-action chess Thompson notebook (see
# that branch for the full derivations): each edge holds a value-belief PMF
# over v = p_win − p_loss ∈ [−1, 1] on a G-point grid plus a scalar mean
# draw-probability, mixture-propagated by P(best) weights (Tesauro), with
# MCTS-Solver proofs as exact spikes. What changes here is WHERE a leaf
# edge's belief comes from:
#
#   before: ONE network forward at the node's own state produced per-action
#           (p_win, p_draw, p_loss, conf) rows out of a 4674-slot head.
#   now:    expanding a node applies each legal action and BATCH-evaluates
#           all resulting child positions in one forward pass. The network
#           only ever predicts the CURRENT state's outcome (3 numbers + a
#           confidence); the child's mover is always the opponent (chess
#           strictly alternates), so the parent-perspective edge belief is
#           the child's prediction with win/loss swapped — an EXACT
#           perspective flip of the Dirichlet (a pure column permutation),
#           not an approximation.
#
# Three structural consequences, all favorable:
#   - every edge is grounded in a real evaluation of the position it leads
#     to (a head-slot never has to rank moves it can't see the result of);
#   - terminal children are detected AT EXPANSION for free — every mate/
#     stalemate/draw one ply below any expanded node enters the tree as an
#     exact MCTS-Solver proof immediately, no simulation needs to stumble
#     into it (mate-in-1 at the root is proven before the first sim);
#   - the NN batch per expansion is ~35 rows instead of 1, which is exactly
#     the large-batch regime the DirectML inference server wants.
# The cost: ~k forwards per expansion instead of 1 — the sim budgets in the
# training cell are cut accordingly (each sim now carries ~k× more
# information; see the hyperparameter block for the arithmetic).

_GRID_G  = 33
_GRID_X  = (np.arange(_GRID_G) + 0.5) / _GRID_G          # (G,) unit-interval grid
_GRID_V  = 2.0 * _GRID_X - 1.0                            # (G,) value-space read-out
_GRID_V2 = _GRID_V ** 2
_GRID_LX  = np.log(_GRID_X)
_GRID_L1X = np.log1p(-_GRID_X)
_SPIKE   = np.eye(_GRID_G)                               # _SPIKE[j] = onehot at j
_SPIKE_WIN, _SPIKE_LOSS, _SPIKE_DRAW = (_SPIKE[_GRID_G - 1], _SPIKE[0],
                                        _SPIKE[_GRID_G // 2])
_VLOSS_PEN = 1.0      # within-wave virtual-loss penalty on a sampled edge value
_MAX_FRAC  = 0.0      # 0 = pure mixture propagation; 1 = hard max


# ── Innovation-based confidence calibration (empirical Bayes — NO caps) ───────
# Unchanged from the per-action notebook (see its tree-ops cell for the full
# derivation): model miscalibration as a variance-inflation factor λ (true
# error variance = λ × claimed), estimated from backup innovations — the
# discrepancy between an edge's held belief and its one-ply-deeper
# replacement (and, via solver proofs, against ground truth) — scored with a
# bounded-influence t-statistic normalized so a calibrated net gives λ → 1
# exactly. Applied once, at leaf construction, scaling Var[v].
_TDOF    = 4.0                              # t dof: bounded outlier influence
_VAR_RES = (2.0 / _GRID_G) ** 2 / 12.0      # per-PMF grid-resolution variance

def _psi(z2):
    return (_TDOF + 1.0) * z2 / (_TDOF + z2)

_zg = np.linspace(-10.0, 10.0, 8001)          # Riemann sum on a uniform grid
_PSI_NORM = float(np.sum(_psi(_zg ** 2)      # (np.trapz was removed in numpy 2)
                         * np.exp(-0.5 * _zg ** 2) / np.sqrt(2.0 * np.pi))
                  * (_zg[1] - _zg[0]))
del _zg


class _Calib:
    """Running empirical-Bayes estimate of the variance-inflation factor
    lambda. observe() takes a squared innovation and the beliefs' claimed
    variance sum; lam() returns the current estimate (1.0 = calibrated)."""
    __slots__ = ('s', 'n', 'd')

    def __init__(self, prior_n=50.0, halflife=2000.0):
        self.s = prior_n
        self.n = prior_n
        self.d = 0.5 ** (1.0 / halflife)

    def observe(self, e2, var_sum):
        z2 = e2 / (var_sum + 2.0 * _VAR_RES)
        self.s = self.s * self.d + _psi(z2) / _PSI_NORM
        self.n = self.n * self.d + 1.0

    def lam(self):
        return self.s / self.n


def _edge_mean_var(node, idx):
    """Mean/variance of an edge's held value belief (as-held, pre-replacement)."""
    p = node.edge[idx]
    m = float(p @ _GRID_V)
    return m, max(float(p @ _GRID_V2) - m * m, 0.0)


def _edge_innovation(node, idx, m0, v0, calib):
    """First-lookahead innovation for a freshly expanded edge, measured AFTER
    its first backup: the edge held the direct child-state evaluation (made
    when `node` was constructed); _backup has just replaced it with the
    flipped mixture over the child's own (grandchild-grounded) edges — or,
    when the expansion PROVED the child, with an exact spike, in which case
    this same read-off scores the old belief against ground truth (m1 =
    ±1/0, v1 = 0). (m0, v0) are the edge's pre-backup mean/variance."""
    m1, v1 = _edge_mean_var(node, idx)
    calib.observe((m0 - m1) ** 2, v0 + v1)


def _beta_pmf_rows(alpha, beta):
    """Discretize Beta(alpha_i, beta_i) onto the unit grid → (k, G) PMF rows.
    Log-space keeps large concentrations (up to CONF_MAX) finite."""
    logw = ((alpha[:, None] - 1.0) * _GRID_LX[None, :]
            + (beta[:, None] - 1.0) * _GRID_L1X[None, :])
    logw -= logw.max(axis=1, keepdims=True)
    w = np.exp(logw)
    return w / w.sum(axis=1, keepdims=True)


def _flip_pmf(pmf):
    """Reflect a value PMF around v=0 (v -> -v): equivalent to reflecting the
    unit grid around 0.5 (x_{G-1-j} = 1 - x_j), since v=2x-1 is affine."""
    return pmf[::-1].copy()


def _dirichlet_leaf_belief(alpha_w, alpha_d, alpha_l, lam=1.0):
    """Dirichlet(alpha_w, alpha_d, alpha_l) evidence → (v-belief PMF (k,G),
    mean draw-probability (k,)) via the exact independent (q,d) decomposition
    (verified numerically in the per-action notebook), moment-matching a Beta
    to the closed-form (E[v], Var[v]). `lam` is the innovation-calibration
    factor: the claimed Var[v] is scaled to the empirically observed error
    scale before the belief is built."""
    C = alpha_w + alpha_d + alpha_l
    Ed = alpha_d / C
    Vd = alpha_d * (alpha_w + alpha_l) / (C**2 * (C + 1.0))
    Eq = alpha_w / (alpha_w + alpha_l)
    Vq = alpha_w * alpha_l / ((alpha_w + alpha_l)**2 * (alpha_w + alpha_l + 1.0))
    EX, VX = 1.0 - Ed, Vd
    EY, VY = 2.0 * Eq - 1.0, 4.0 * Vq
    Ev = EX * EY
    Vv = EX**2 * VY + EY**2 * VX + VX * VY
    mu01  = (Ev + 1.0) / 2.0
    var01 = np.clip(Vv / 4.0 * lam, 1e-9, None)
    var01 = np.minimum(var01, mu01 * (1.0 - mu01) * 0.999)   # below max-possible
    conc  = np.maximum(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    a_beta = np.maximum(mu01 * conc, ALPHA_FLOOR)
    b_beta = np.maximum((1.0 - mu01) * conc, ALPHA_FLOOR)
    return _beta_pmf_rows(a_beta, b_beta), Ed


def _prob_best(E, C):
    """P(edge i is the argmax) for each edge, from PMFs `E` (k,G) and their
    inclusive CDFs `C` (k,G): sum_x p_i(x) * prod_{j!=i} P(X_j < x), splitting
    ties at x half-and-half. Leave-one-out product done in log-space."""
    logG = np.log(np.clip(C - 0.5 * E, 1e-12, 1.0))     # log P(X_j < x), tie-split
    loo  = np.exp(logG.sum(axis=0)[None, :] - logG)     # prod_{j!=i} (k, G)
    w = (E * loo).sum(axis=1)                           # (k,)
    s = w.sum()
    return w / s if s > 0 else np.full(len(w), 1.0 / len(w))


class _TNode:
    """One expanded state. `edge[i]` is the value-belief PMF for action i (from
    this node's mover's perspective); `draw[i]` is the mean draw-probability for
    action i. `term[i] >= 0` marks a PROVEN outcome for edge i.

    Takes per-legal-action `p3` (k, 3) outcome probabilities and `conf` (k,)
    prior strengths ALREADY IN THE MOVER'S PERSPECTIVE — in this notebook
    those rows come from evaluating each CHILD state and swapping the
    win/loss columns (see _make_node), not from a per-action network head."""
    __slots__ = ('player', 'legal', 'edge', 'draw', 'vloss', 'term', 'children',
                 'obs')

    def __init__(self, player, legal, p3, conf, lam=1.0):
        self.player   = player
        self.legal    = np.asarray(legal, dtype=np.int32)
        self.obs      = None      # fp16 observation, stashed at expansion time
        p  = np.asarray(p3,   dtype=np.float64)
        cf = np.asarray(conf, dtype=np.float64)
        aw = np.maximum(cf * p[:, 0], ALPHA_FLOOR)
        ad = np.maximum(cf * p[:, 1], ALPHA_FLOOR)
        al = np.maximum(cf * np.clip(p[:, 2], 0.0, None), ALPHA_FLOOR)
        self.edge, self.draw = _dirichlet_leaf_belief(aw, ad, al, lam)
        self.vloss    = np.zeros(len(legal), dtype=np.int32)
        self.term     = np.full(len(legal), -1, dtype=np.int8)
        self.children = [None] * len(legal)


# Proven outcomes are baked INTO the edge arrays at the moment term is set
# (_set_term is the ONLY way engine code marks a proven edge): the edge PMF
# becomes the exact spike and draw the exact 0/1. Indexed by outcome code.
_TERM_PMF  = np.stack([_SPIKE_WIN, _SPIKE_DRAW, _SPIKE_LOSS])
_TERM_DRAW_VAL = np.array([0.0, 1.0, 0.0])
_TERM_P3   = np.eye(3)                     # onehot (win, draw, loss) rows


def _set_term(node, idx, outcome):
    node.term[idx] = outcome
    node.edge[idx] = _TERM_PMF[outcome]
    node.draw[idx] = _TERM_DRAW_VAL[outcome]


def _effective_edges(node):
    """Edge value-belief PMFs; proven rows are already exact spikes."""
    return node.edge


def _effective_draw(node):
    """Edge mean-draw-probabilities; proven rows already exact (_set_term)."""
    return node.draw


def _node_beliefs(node):
    """A node's own (value PMF, mean draw-probability) = the MIXTURE over its
    edges weighted by each edge's probability of being the best reply (Tesauro
    mixture propagation), computed ONCE and reused for both quantities."""
    E = _effective_edges(node)
    D = _effective_draw(node)
    if E.shape[0] == 1:
        return E[0].copy(), float(D[0])
    C = np.cumsum(E, axis=1)
    np.clip(C, 0.0, 1.0, out=C)
    w = _prob_best(E, C)                                # (k,) P(edge best)
    mix = w @ E                                          # (G,) value mixture
    mean_draw = float(w @ D)                              # scalar draw mixture
    if _MAX_FRAC > 0.0:                                  # blend toward max dist
        Fmax = C.prod(axis=0)
        mx = np.empty(_GRID_G)
        mx[0]  = Fmax[0]
        mx[1:] = np.diff(Fmax)
        np.maximum(mx, 0.0, out=mx)
        ms = mx.sum()
        mix = (1.0 - _MAX_FRAC) * mix + _MAX_FRAC * (mx / ms if ms > 0 else mix)
    s = mix.sum()
    return (mix / s if s > 0 else np.full(_GRID_G, 1.0 / _GRID_G)), mean_draw


def _sample_edge_values(node, rng):
    """One Thompson sample of each edge's value, less a virtual-loss penalty for
    in-flight edges (within-wave diversification only — beliefs are untouched)."""
    C = np.cumsum(_effective_edges(node), axis=1)
    u = rng.random_sample(len(node.legal))
    idx = np.minimum((C < u[:, None]).sum(axis=1), _GRID_G - 1)   # inverse-CDF
    v = _GRID_V[idx].copy()
    if node.vloss.any():
        v -= _VLOSS_PEN * node.vloss
    return v


def _select_leaf(root, root_state, rng):
    """Descend by Thompson sampling until an unexpanded or proven edge.
    Applies virtual loss along the path. Returns (path, leaf_state, edge):
      proven edge   → (path, None, None)  — outcome already in node.term
      unexpanded    → (path, state-at-leaf, (node, idx))  — needs expansion
    path is a list of (node, action_idx). Unlike the per-action notebook,
    selection can never DISCOVER a terminal: every terminal child was already
    marked proven when its parent was expanded (see _expand_children), so a
    non-proven edge is guaranteed to lead to a live position — the state is
    only cloned/advanced along edges actually descended."""
    node  = root
    state = root_state.clone()
    path  = []
    while True:
        idx = int(_sample_edge_values(node, rng).argmax())
        node.vloss[idx] += 1
        path.append((node, idx))
        if node.term[idx] >= 0:
            return path, None, None
        state.apply_action(int(node.legal[idx]))
        child = node.children[idx]
        if child is None:
            return path, state, (node, idx)
        node = child


def _backup(path):
    """Remove the virtual loss and RECOMPUTE each expanded edge on the path from
    its child's current belief (bottom-up), so the fresh expansion's evidence is
    mixture-propagated to the root. Value flips sign going up a ply (opponent's
    move); draw-probability does NOT flip. Nothing is accumulated."""
    for node, idx in reversed(path):
        node.vloss[idx] -= 1
        if node.term[idx] < 0 and node.children[idx] is not None:
            vpmf, mdraw = _node_beliefs(node.children[idx])
            node.edge[idx] = _flip_pmf(vpmf)
            node.draw[idx] = mdraw


# ── Node expansion: batch-evaluate ALL child states ───────────────────────────

def _expand_children(state):
    """Apply every legal action of `state` once. Terminal children yield their
    exact outcome (already from THIS state's mover's perspective — no flip
    needed later); live children yield one fp16 observation row (from the
    child mover's perspective, which is what the network expects) for the NN
    batch. Returns (legal, term (k,) int8, rows [fp16 (1280,)],
    row_map int32 (len(rows),)) with row_map[j] = the edge index of row j."""
    legal = state.legal_actions()
    me    = state.current_player()
    term  = np.full(len(legal), -1, dtype=np.int8)
    rows, row_map = [], []
    for i, a in enumerate(legal):
        ch = state.clone()
        ch.apply_action(int(a))
        if ch.is_terminal():
            r = ch.returns()[me]
            term[i] = _WIN if r > 0 else (_LOSS if r < 0 else _DRAW)
        else:
            rows.append(np.asarray(ch.observation_tensor(ch.current_player()),
                                   dtype=np.float16))
            row_map.append(i)
    return legal, term, rows, np.asarray(row_map, dtype=np.int32)


def _make_node(player, legal, term, p3_rows, conf_rows, row_map, lam=1.0):
    """Assemble a _TNode from an expansion's evaluated child rows. `p3_rows`
    (n_live, 3) are the network's (win, draw, loss) predictions for the LIVE
    child states — in the CHILD mover's perspective, so each row is reversed
    ([::-1] swaps win<->loss, draw stays put: the exact Dirichlet perspective
    flip, since the child's mover is always this node's opponent). Terminal
    edges (term >= 0) get exact proof spikes via _set_term; their belief rows
    are placeholders that the spikes overwrite."""
    k  = len(legal)
    p3 = np.full((k, 3), 1.0 / 3.0, dtype=np.float64)
    cf = np.ones(k, dtype=np.float64)
    if len(row_map):
        p3[row_map] = np.asarray(p3_rows, dtype=np.float64)[:, ::-1]
        cf[row_map] = np.asarray(conf_rows, dtype=np.float64)
    node = _TNode(player, legal, p3, cf, lam)
    for i in np.nonzero(term >= 0)[0]:
        _set_term(node, int(i), int(term[i]))
    return node


# ── MCTS-Solver: propagate proven wins/draws/losses up the tree ───────────────
# Terminal children are marked proven at EXPANSION time (see _expand_children),
# so proofs enter the tree a ply earlier than in the per-action notebook —
# a node with a mate-in-1 is solved the moment it is constructed. From there
# the same WIN > DRAW > LOSS precedence propagates upward.

def _node_solved_outcome(node):
    """The proven outcome for node.player if the node is solved, else None. WIN
    as soon as any edge is a proven win; else DRAW if any edge is a proven draw
    and all edges are proven; LOSS only once every edge is a proven loss."""
    t = node.term
    if (t == _WIN).any():
        return _WIN
    if (t >= 0).all():
        return _DRAW if (t == _DRAW).any() else _LOSS
    return None


def _solver_sweep(path, aux=None, conf_cap=None):
    """Walk the path deepest-edge-first; whenever the CHILD under a path edge
    has become solved, prove that edge (flipped — the parent's mover is the
    opponent) and continue upward. Stops at the first unsolved child or
    already-proven edge. Called after an expansion's backup: the freshly
    created child under path[-1] may be born solved (terminal children are
    proofs at construction), and each proof can complete its parent in turn.

    Solver-labeled auxiliary samples: when `aux` (a list) is given, each
    NEWLY-solved node is emitted as a training sample
    (node.obs, state_target(node, conf_cap)) — exact game-theoretic labels,
    the strongest supervision this system can produce. The emission point is
    the node's parent edge transitioning to proven, which happens exactly
    ONCE per node — built-in deduplication."""
    for k in range(len(path) - 1, -1, -1):
        parent, idx = path[k]
        child = parent.children[idx]
        if child is None:
            break
        out = _node_solved_outcome(child)
        if out is None or parent.term[idx] >= 0:
            break
        _set_term(parent, idx, int(_FLIP_TERM[out]))
        if aux is not None and child.obs is not None:
            aux.append((child.obs, state_target(child, conf_cap)))


def state_target(node, conf_cap):
    """A node's posterior belief → DENSE (3,) fp32 training counts for the
    state-value network: the node's own mixture belief (value PMF + mean
    draw-probability) moment-matched back to (p_win, p_draw, p_loss) and a
    concentration (from Var[v] — the same reconstruction the per-action
    notebook applied per edge, applied once to the node itself). A SOLVED
    node gets the exact certain target: full `conf_cap` on the proven
    outcome. Concentration is capped at `conf_cap` (mean-preserving) so no
    single generation teaches unbounded certainty.

    Note the self-reference question resolves differently here: every edge
    belief in the mixture is grounded in a real evaluation of a real child
    position (or a proof) — there is no 'echoed unexpanded prior' to
    exclude, because unexpanded edges don't exist (expansion evaluates all
    children at once)."""
    out = _node_solved_outcome(node)
    if out is not None:
        cnt = np.full(3, ALPHA_FLOOR, dtype=np.float32)
        cnt[out] = conf_cap - 2.0 * ALPHA_FLOOR   # total exactly conf_cap
        return cnt
    vpmf, d = _node_beliefs(node)
    mean_v = float(vpmf @ _GRID_V)
    var_v  = max(float(vpmf @ _GRID_V2) - mean_v * mean_v, 1e-9)
    mu01   = (mean_v + 1.0) / 2.0
    var01  = max(var_v / 4.0, 1e-9)
    conc   = max(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    conc   = min(conc, conf_cap)                       # cap certainty growth
    nondraw = min(max(1.0 - d, 0.0), 1.0)
    p_win  = min(max((mean_v + nondraw) / 2.0, 0.0), nondraw)
    p_loss = nondraw - p_win
    return (np.array([p_win, d, p_loss]) * conc).astype(np.float32)


def z_mix_episode(samples, returns, z_mix, z_gamma=1.0):
    """Blend the OBSERVED game outcome into each recorded sample's state
    target (proportion-average, concentration preserved):

        counts  ←  (1 − w)·counts  +  w·total·onehot(z)

    e.g. w=0.5, counts (10,30,60), game lost → (5,15,80). The per-sample
    weight is RAMPED by distance from the game's end:
    `w = z_mix · z_gamma^(N−1−i)` for sample i of N (a TD(λ)-style
    eligibility trace) — z is a near-noiseless label two plies from the end
    and mostly noise 80 plies out.

    For STATE targets the blend needs no played-action caveat: the final
    outcome is one real multinomial observation of THIS position's outcome
    under the current policy — exactly the quantity the state head predicts.
    (The per-action notebook had to restrict z to the played action because
    z said nothing about siblings; a state target has no siblings.)

    `samples` carry transient per-move metadata (obs, counts, mover, proven);
    `returns` is state.returns() of a game that actually FINISHED — never
    call this for a ply-capped game (see strip_episode_meta). Solver-PROVEN
    targets are left untouched (a proof is strictly stronger evidence than
    one noisy outcome). Returns standard (obs, counts) training pairs."""
    out = []
    n = len(samples)
    for i, (obs, tc, player, proven) in enumerate(samples):
        w = z_mix * (z_gamma ** (n - 1 - i))
        if w > 0.0 and not proven:
            r = returns[player]
            col = _WIN if r > 0 else (_LOSS if r < 0 else _DRAW)
            tc = tc.copy()
            conc = float(tc.sum())
            tc *= (1.0 - w)
            tc[col] += w * conc
        out.append((obs, tc))
    return out


def strip_episode_meta(samples):
    """Ply-capped games: drop the z-mix metadata WITHOUT mixing — the eventual
    outcome was never observed, and the pure-search targets stay fully valid
    (this is what keeps early cutoff safe in this framework)."""
    return [(o, tc) for o, tc, _p, _v in samples]


# ── Backward-restart curriculum (unchanged from the per-action notebook) ──────
# Restart a fraction of self-play games a few plies before endings harvested
# from the net's OWN decisive / high-innovation games, by ACTION-REPLAY (not
# FEN — threefold repetition depends on history a FEN omits). Sound because
# no target depends on where a game started.

def restart_prefix(seq, rng, k_min, k_max):
    """The prefix of a stored action sequence to replay for a restart: drop a
    random k ∈ [k_min, k_max] tail plies. None if the sequence is too short."""
    if len(seq) <= k_min:
        return None
    k = int(rng.randint(k_min, min(k_max, len(seq) - 1) + 1))
    return seq[:len(seq) - k]


def _root_value(node):
    """Scalar posterior value of a node (mixture mean over its value belief) —
    used to measure the per-move root value swing for high-innovation harvest."""
    vpmf, _ = _node_beliefs(node)
    return float(vpmf @ _GRID_V)


def _descend(root, action):
    """Tree reuse: after `action` is played, the subtree under it becomes the
    next search's root — proofs, expansions, and refined beliefs all carry
    over. Safe by construction: beliefs are recomputed from the current
    subtree on every backup, and virtual losses are always unwound before a
    move is played. Returns None when the edge was never expanded."""
    if root is None:
        return None
    hit = np.nonzero(root.legal == action)[0]
    return root.children[int(hit[0])] if len(hit) else None


def root_pick(root, rng, thompson):
    """Final move choice: Thompson-sample the value beliefs (exploratory phase)
    or take the belief-mean argmax (deterministic endgame / evaluation)."""
    if thompson:
        v = _sample_edge_values(root, rng)
    else:
        v = _effective_edges(root) @ _GRID_V             # posterior value means
    return int(root.legal[int(v.argmax())])


# ── Batched child-eval Thompson-MCTS bot ──────────────────────────────────────

def _expand_state_node(network, device, state, lam=1.0):
    """Expand one state into a _TNode with a single forward pass over all its
    live child positions. Used for search roots and small-batch callers; the
    self-play loops batch MANY expansions into one pass instead."""
    legal, term, rows, row_map = _expand_children(state)
    if len(row_map):
        x = batch_to_tensor(np.stack(rows), device)
        with torch.no_grad():
            p, c = network(x)
        p_np, c_np = p.cpu().numpy(), c.cpu().numpy()
    else:
        p_np = np.zeros((0, 3), dtype=np.float32)
        c_np = np.zeros(0, dtype=np.float32)
    node = _make_node(state.current_player(), legal, term, p_np, c_np,
                      row_map, lam)
    node.obs = np.asarray(state.observation_tensor(state.current_player()),
                          dtype=np.float16)
    return node


class ThompsonMCTSBot:
    """mcts_search(state) → root _TNode whose `.edge`/`.draw` hold the value/
    draw beliefs. Collects up to `batch_size` leaf EXPANSIONS per wave
    (virtual loss + Thompson randomness keep waves diverse) and evaluates all
    their child positions in one NN forward pass (~35·batch_size rows).
    Weights are read live from `network`."""

    def __init__(self, game, network, device, max_simulations,
                 batch_size=16, random_state=None, calib=None):
        self.game            = game
        self.network         = network
        self.device          = device
        self.max_simulations = max_simulations
        self.batch_size      = batch_size
        self._rng            = random_state or np.random.RandomState()
        # Innovation calibration persists ACROSS searches: miscalibration is a
        # property of the net, not of one position (see _Calib).
        self.calib           = calib if calib is not None else _Calib()

    def mcts_search(self, state):
        root = _expand_state_node(self.network, self.device, state,
                                  self.calib.lam())
        sims = 0
        while sims < self.max_simulations:
            if _node_solved_outcome(root) is not None:
                break                          # MCTS-Solver: root proven, stop
            wave    = min(self.batch_size, self.max_simulations - sims)
            pending = []                       # (path, leaf_state, (node, idx))
            for _ in range(wave):
                path, st, edge = _select_leaf(root, state, self._rng)
                if st is None:                 # proven edge re-selected
                    _backup(path)
                    sims += 1
                else:
                    pending.append((path, st, edge))

            # Expand every unique leaf edge: ALL their child positions in ONE
            # forward pass.
            unique = {}
            for path, st, (node, idx) in pending:
                unique.setdefault((id(node), idx), (node, idx, st, path))
            jobs, rows = [], []
            for node, idx, st, path in unique.values():
                legal, term, rws, rmap = _expand_children(st)
                jobs.append((node, idx, st, path, legal, term, rmap, len(rws)))
                rows.extend(rws)
            p_np = np.zeros((0, 3), dtype=np.float32)   # all-terminal wave: no rows
            c_np = np.zeros(0, dtype=np.float32)
            if rows:
                x = batch_to_tensor(np.stack(rows), self.device)
                with torch.no_grad():
                    p, c = self.network(x)
                p_np, c_np = p.cpu().numpy(), c.cpu().numpy()
            lam, off, pre = self.calib.lam(), 0, []
            for node, idx, st, path, legal, term, rmap, nr in jobs:
                child = _make_node(st.current_player(), legal, term,
                                   p_np[off:off + nr], c_np[off:off + nr],
                                   rmap, lam)
                off += nr
                child.obs = np.asarray(
                    st.observation_tensor(st.current_player()),
                    dtype=np.float16)
                pre.append(_edge_mean_var(node, idx))   # pre-backup belief
                node.children[idx] = child
            # Mixture-propagate each fresh expansion up its path.
            for path, st, (node, idx) in pending:
                _backup(path)
                sims += 1
            # Solver proofs (a child born solved proves its parent edge), then
            # innovations read off the final backed-up/proven edges.
            for (node, idx, st, path, *_), (m0, v0) in zip(jobs, pre):
                _solver_sweep(path)
                _edge_innovation(node, idx, m0, v0, self.calib)
        return root


def make_thompson_bot(game, network, device, num_simulations, batch_size=16):
    return ThompsonMCTSBot(game, network, device, num_simulations,
                           batch_size=batch_size)


# ── Evaluation ─────────────────────────────────────────────────────────────────

def _eval_children(network, state, device):
    """One-ply value look: batch-evaluate every legal move's resulting
    position → (legal int32 (k,), p3 (k,3) MOVER-perspective, conf (k,),
    term (k,)). Terminal moves get their exact outcome (a mate-in-1 is
    always visible here — the search-free player natively plays mates and
    avoids stalemates, which the old per-action policy head could only
    learn statistically)."""
    legal, term, rows, row_map = _expand_children(state)
    p3 = np.full((len(legal), 3), 1.0 / 3.0)
    cf = np.ones(len(legal))
    if len(row_map):
        x = batch_to_tensor(np.stack(rows), device)
        with torch.no_grad():
            p, c = network(x)
        p3[row_map] = p.cpu().numpy()[:, ::-1]          # flip to mover's view
        cf[row_map] = c.cpu().numpy()
    ti = np.nonzero(term >= 0)[0]
    if len(ti):
        p3[ti] = _TERM_P3[term[ti]]     # term already in mover's perspective
        cf[ti] = CONF_MAX
    return np.asarray(legal, dtype=np.int32), p3, cf, term


def policy_action(network, state, device, sample=False, rng=None):
    """Search-free move from the raw network: one batched forward over all
    child positions, then argmax of the mover-perspective mean value
    (p_win - p_loss), or (sample=True) one Thompson sample from each child's
    predicted Dirichlet — the search-free analogue of temperature sampling."""
    legal, p3, cf, _term = _eval_children(network, state, device)
    if sample:
        rng = rng or np.random
        a = np.maximum(cf[:, None] * p3, ALPHA_FLOOR)   # (k,3)
        g = rng.standard_gamma(a)
        p = g / (g.sum(axis=1, keepdims=True) + 1e-12)
        v = p[:, _WIN] - p[:, _LOSS]
    else:
        v = p3[:, _WIN] - p3[:, _LOSS]
    return int(legal[int(v.argmax())])


# Evaluation-side games have no equivalent of self-play's MAX_SELFPLAY_PLIES:
# chess is only EVENTUALLY bounded (the 50-move rule), and an early,
# undertrained or weak net can go hundreds of plies without a capture or pawn
# move before that resets. Cap it and score a cut-off game as a draw.
MAX_EVAL_PLIES = 300


def play_match(net_a, net_b, game, n_games, device):
    """net_a vs net_b over n_games, alternating colours. net == None ==> random.
    Search-free, Thompson-sampled moves (per-game variety without extra noise).
    Returns (wins_a, draws, wins_b) from net_a's view — the quick progress
    pulse (does not touch Elo)."""
    wa = dd = wb = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2
        ply = 0
        while not state.is_terminal() and ply < MAX_EVAL_PLIES:
            net = net_a if state.current_player() == a_player else net_b
            if net is None:
                action = random.choice(state.legal_actions())
            else:
                action = policy_action(net, state, device, sample=True)
            state.apply_action(action)
            ply += 1
        if not state.is_terminal():
            dd += 1                      # ply cap hit: score as a draw
            continue
        r = state.returns()[a_player]
        if r > 0:   wa += 1
        elif r < 0: wb += 1
        else:       dd += 1
    return wa, dd, wb


def _mcts_move(bot, state):
    """Posterior-mean argmax after a full search (the in-tree Thompson sampling
    gives the per-game variety that keeps a round-robin from replaying
    identical games)."""
    return root_pick(bot.mcts_search(state), bot._rng, thompson=False)


# ── Benchmark-pool evaluation: running-Elo tournaments (no fixed anchor) ─────────
# Identical structure to the earlier notebooks: every net is a tournament
# player entering at START_ELO, updated with the Elo K-factor, in independent
# TRACKS (search-free 1-ply value and one per MCTS budget) with per-track Elo
# tables and head-to-head matrices.

import math
import itertools
from collections import defaultdict


def run_tournament(players, elos, wdl, game, device,
                   games_per_pair=10, k=32.0, sims=0, opening_plies=0,
                   focus_label=None, refresh_pairs=0):
    """One full round-robin for a single track. Moves are chosen search-free
    (sims == 0: one-ply value argmax) or by Thompson-MCTS with `sims`
    simulations (posterior-mean argmax). The first `opening_plies` moves of
    each game are random (both sides) only to vary games; strength is then
    measured noise-free. Every unordered pair plays `games_per_pair` games,
    half per colour, all shuffled into random order; Elo is updated after
    each game. `elos` must already hold a rating for every label."""
    net  = {p['label']: p['net'] for p in players}
    bots = {}
    if sims > 0:
        bs = max(1, min(sims, 16))
        for p in players:
            if p['net'] is not None:
                bots[p['label']] = make_thompson_bot(game, p['net'], device,
                                                     sims, bs)

    def pick(label, state):
        if net[label] is None:
            return random.choice(state.legal_actions())
        if sims <= 0:
            return policy_action(net[label], state, device, sample=False)
        return _mcts_move(bots[label], state)

    pairs = list(itertools.combinations(net.keys(), 2))
    if focus_label is not None:
        # Linear-cost mode: every pair involving the newest model, plus a few
        # random old-vs-old pairs so earlier ratings keep mixing.
        focus  = [p for p in pairs if focus_label in p]
        others = [p for p in pairs if focus_label not in p]
        random.shuffle(others)
        pairs = focus + others[:refresh_pairs]
    schedule = []
    for a, b in pairs:
        half = games_per_pair // 2
        schedule += [(a, b)] * half
        schedule += [(b, a)] * half
        schedule += [(a, b)] * (games_per_pair - 2 * half)   # odd remainder
    random.shuffle(schedule)

    for p0, p1 in schedule:
        state = game.new_initial_state()
        ply = 0
        while not state.is_terminal() and ply < MAX_EVAL_PLIES:
            if ply < opening_plies:
                action = random.choice(state.legal_actions())  # variety only
            else:
                lab = p0 if state.current_player() == 0 else p1
                action = pick(lab, state)
            state.apply_action(action)
            ply += 1
        # A non-terminal state's returns() is exactly [0.0, 0.0] in chess, so a
        # ply-cap cutoff scores as a draw here with no special-casing needed.
        r  = state.returns()[0]
        s0 = 1.0 if r > 0 else (0.0 if r < 0 else 0.5)
        e0 = 1.0 / (1.0 + 10 ** ((elos[p1] - elos[p0]) / 400.0))
        elos[p0] += k * (s0 - e0)
        elos[p1] += k * ((1.0 - s0) - (1.0 - e0))
        cell = wdl[(p0, p1)]
        if r > 0:   cell[0] += 1
        elif r < 0: cell[2] += 1
        else:       cell[1] += 1

In [ ]:
# ── Parallel self-play: many games, one forward pass ────────────────────────────
# ThompsonMCTSBot batches expansions within ONE search. The next multiplier is
# batching ACROSS games: run N games concurrently, collect every game's
# expansion wave, and evaluate ALL their child positions in a single NN
# forward pass — with the state-value design each expansion contributes ~35
# rows, so the joint batches are an order of magnitude larger than the
# per-action notebook's (exactly what the GPU wants).
import os


class ThompsonParallelSelfPlay:
    """Interleaves the searches of n_parallel self-play games so their
    expansion waves (roots and leaves alike) share one NN forward pass over
    all child positions. Weights are read live from `network`, so games in
    flight pick up training updates immediately."""

    def __init__(self, game, network, device, n_parallel=8, wave_per_game=4,
                 fast_sims=64, full_sims=256, fast_prob=0.75, temp_threshold=20,
                 conf_cap=100.0, seed=None,
                 pool_prob=0.0, checkpoint_dir=None, channels=None,
                 num_blocks=None, max_selfplay_plies=400, z_mix=0.5,
                 z_gamma=1.0, restart_prob=0.0, restart_k_min=2,
                 restart_k_max=30, restart_pool_cap=128, swing_thresh=0.6,
                 random_pool_frac=0.5):
        self.game = game
        self.network = network
        self.device = device
        self.n_parallel = n_parallel
        self.wave = wave_per_game
        self.fast_sims = fast_sims
        self.full_sims = full_sims
        self.fast_prob = fast_prob
        self.temp_threshold = temp_threshold
        self.conf_cap = conf_cap
        # Self-play safety valve only — a cut-off game's samples keep their
        # pure search targets (z_mix applies only to games that actually
        # finish — see strip_episode_meta), so nothing recorded is invalidated.
        self.max_selfplay_plies = max_selfplay_plies
        self.z_mix = z_mix        # observed-outcome blend weight (see cell 5)
        self.z_gamma = z_gamma    # end-of-game ramp on the z-mix weight
        self._rng = np.random.RandomState(seed)
        # Backward-restart curriculum (see cell 5): seed pool of action
        # sequences from decisive / high-innovation games; FIFO-capped.
        self.restart_prob = restart_prob
        self.restart_k_min = restart_k_min
        self.restart_k_max = restart_k_max
        self.restart_pool_cap = restart_pool_cap
        self.swing_thresh = swing_thresh
        self.random_pool_frac = random_pool_frac
        self._restart_pool = []
        # Opponent diversity (see _resolve_pool_moves): some fraction of games
        # have one side played by a frozen historical checkpoint (or a uniform-
        # random mover) instead of mirror-matching the live net against itself.
        # Only the live side's moves become training samples.
        self.pool_prob = pool_prob
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}          # benchmark label -> frozen CPU net
        # Innovation calibration (see cell 5): one persistent estimator for
        # this self-play loop; its lambda scales every belief built here.
        self.calib = _Calib()
        self.last_lam = self.calib.lam()
        self.last_aux = 0             # solver-labeled samples in last episode
        self.last_restart_pool = 0
        # Cumulative self-play diagnostics (the training loop diffs these each
        # eval window): games produced, and how they ended.
        self.stats = {'games': 0, 'draw': 0, 'cutoff': 0, 'decisive': 0,
                      'plies': 0}
        self.slots = [self._new_game() for _ in range(n_parallel)]

    def _push_seed(self, seq):
        if len(seq) >= 2:
            self._restart_pool.append(list(seq))
            if len(self._restart_pool) > self.restart_pool_cap:
                del self._restart_pool[0]

    def _load_pool_net(self, label):
        net = self._pool_nets.get(label)
        if net is None:
            path = os.path.join(self.checkpoint_dir, f'bench_{label}.pt')
            net = ThompsonValueChessNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[label] = net
        return net

    def _new_game(self):
        # Playout cap randomization is chosen per game, as in serial self-play.
        sims = (self.fast_sims if self._rng.rand() < self.fast_prob
                else self.full_sims)
        rng = self._rng
        state = self.game.new_initial_state()
        actions = []
        # Restart: replay a harvested sequence up to a few plies before its end.
        if self._restart_pool and rng.rand() < self.restart_prob:
            seq = self._restart_pool[rng.randint(len(self._restart_pool))]
            pref = restart_prefix(seq, rng, self.restart_k_min,
                                  self.restart_k_max)
            if pref:
                st = self.game.new_initial_state()
                ok = True
                for a in pref:
                    if st.is_terminal() or a not in st.legal_actions():
                        ok = False
                        break
                    st.apply_action(int(a))
                if ok and not st.is_terminal():
                    state, actions = st, list(pref)
        slot = {'state': state, 'hist': [], 'aux': [], 'actions': actions,
                'move': 0, 'sims': sims, 'root': None, 'n': 0, 'pool': None,
                'root_v0': None, 'best_swing': -1.0, 'best_len': 0}
        if self.pool_prob > 0 and rng.rand() < self.pool_prob:
            try:
                labels = [f[6:-3] for f in os.listdir(self.checkpoint_dir)
                          if f.startswith('bench_') and f.endswith('.pt')] \
                    if self.checkpoint_dir else []
            except OSError:
                labels = []
            if not labels or rng.rand() < self.random_pool_frac:
                slot['pool'] = {'label': 'random',
                                'side': int(rng.randint(0, 2)), 'net': None}
            else:
                label = labels[rng.randint(len(labels))]
                slot['pool'] = {'label': label,
                                'side': int(rng.randint(0, 2)),
                                'net': self._load_pool_net(label)}
        return slot

    def _resolve_pool_moves(self):
        """Resolve any pending frozen-opponent moves. A 'random' opponent plays
        a uniform legal move (no NN eval); any other label does one child-batch
        eval + greedy value argmax (which natively plays mates — terminal
        children are exact). No search tree either way — these moves are never
        training targets. Returns finished episodes (a pool move can end the
        game), matching _step()'s return contract."""
        done = []
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            if pool['label'] == 'random':
                legal = state.legal_actions()
                action = int(legal[self._rng.randint(len(legal))])
            else:
                action = policy_action(pool['net'], state, 'cpu', sample=False)
            # Keep the live side's reused tree in sync with the opponent's move.
            s['root'] = _descend(s['root'], action)
            state.apply_action(action)
            s['actions'].append(action)
            s['move'] += 1
            if state.is_terminal() or s['move'] >= self.max_selfplay_plies:
                done.append(self._finish_slot(i))
        return done

    def _finish_slot(self, i):
        """Harvest restart seeds, resolve z-mix (or strip), record per-game
        diagnostics, reset the slot, return the episode's training samples."""
        s = self.slots[i]
        st = s['state']
        term = st.is_terminal()
        if term:
            ret = st.returns()
            hist2 = z_mix_episode(s['hist'], ret, self.z_mix, self.z_gamma)
            result = 'draw' if (ret[0] == 0.0 and ret[1] == 0.0) else 'decisive'
            if self.restart_prob > 0 and result == 'decisive':
                self._push_seed(s['actions'])            # decisive game
        else:
            hist2 = strip_episode_meta(s['hist'])
            result = 'cutoff'
        if self.restart_prob > 0 and s['best_swing'] >= self.swing_thresh:
            self._push_seed(s['actions'][:s['best_len']])
        self.last_aux = len(s['aux'])
        self.last_restart_pool = len(self._restart_pool)
        self.stats['games'] += 1
        self.stats[result] += 1
        self.stats['plies'] += int(s['move'])
        self.slots[i] = self._new_game()
        return hist2 + s['aux']

    def episodes(self):
        """Infinite generator of finished episodes: lists of (obs, counts)."""
        while True:
            for data in self._step():
                self.last_lam = self.calib.lam()
                yield data

    def _step(self):
        """Advance every game by one expansion wave (one joint forward pass
        over all child positions); returns episodes that finished this step."""
        rng     = self._rng
        done    = self._resolve_pool_moves()   # frozen-opponent moves first
        pending = []            # (slot_idx, path) — backups owed this wave
        jobs    = []            # expansion jobs (root or leaf), in rows order
        rows    = []            # concatenated child-observation rows
        seen    = set()         # unique leaf edges already queued this wave
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is not None and s['state'].current_player() == pool['side']:
                continue         # next move belongs to the frozen opponent
            if s['root'] is None:
                legal, term, rws, rmap = _expand_children(s['state'])
                jobs.append(('root', i, None, None, s['state'], legal, term,
                             rmap, len(rws)))
                rows.extend(rws)
                continue         # root expands this step; leaves start next step
            if _node_solved_outcome(s['root']) is not None:
                continue         # MCTS-Solver: root proven — finalized below
            wave = min(self.wave, s['sims'] - s['n'])
            for _ in range(max(wave, 0)):
                path, st, edge = _select_leaf(s['root'], s['state'], rng)
                if st is None:                 # proven edge re-selected
                    _backup(path)
                    s['n'] += 1
                    continue
                node, idx = edge
                pending.append((i, path))
                if (id(node), idx) not in seen:   # expansion deduped; every
                    seen.add((id(node), idx))     # pending path still backs up
                    legal, term, rws, rmap = _expand_children(st)
                    jobs.append(('leaf', i, node, idx, st, legal, term,
                                 rmap, len(rws)))
                    rows.extend(rws)

        # ONE forward pass for every job's child positions, across all games.
        p_np = np.zeros((0, 3), dtype=np.float32)
        c_np = np.zeros(0, dtype=np.float32)
        if rows:
            x = batch_to_tensor(np.stack(rows), self.device)
            with torch.no_grad():
                p, c = self.network(x)
            p_np, c_np = p.cpu().numpy(), c.cpu().numpy()
        lam, off = self.calib.lam(), 0
        fresh = []                # (node, idx, m0, v0, path) for leaf jobs
        for kind, i, node, idx, st, legal, term, rmap, nr in jobs:
            child = _make_node(st.current_player(), legal, term,
                               p_np[off:off + nr], c_np[off:off + nr],
                               rmap, lam)
            off += nr
            child.obs = np.asarray(st.observation_tensor(st.current_player()),
                                   dtype=np.float16)
            if kind == 'root':
                self.slots[i]['root'] = child
                self.slots[i]['root_v0'] = _root_value(child)  # prior value
            else:
                m0, v0 = _edge_mean_var(node, idx)
                node.children[idx] = child
                fresh.append((i, node, idx, m0, v0))
        for i, path in pending:
            _backup(path)                       # mixture-propagate the wave
            self.slots[i]['n'] += 1
        # Solver proofs + innovations, read off the backed-up/proven edges.
        fresh_paths = {}                        # (id(node), idx) -> path
        for i, path in pending:
            fresh_paths.setdefault((id(path[-1][0]), path[-1][1]), path)
        for i, node, idx, m0, v0 in fresh:
            path = fresh_paths.get((id(node), idx))
            if path is not None:
                _solver_sweep(path, self.slots[i]['aux'], self.conf_cap)
            _edge_innovation(node, idx, m0, v0, self.calib)

        # Finalize finished (or solver-proven, or ply-capped) searches; emit
        # completed episodes.
        for i, s in enumerate(self.slots):
            if s['root'] is None:
                continue
            if s['n'] < s['sims'] and _node_solved_outcome(s['root']) is None:
                continue
            self._play_move(i)
            if s['state'].is_terminal() or s['move'] >= self.max_selfplay_plies:
                done.append(self._finish_slot(i))
        return done

    def _play_move(self, i):
        """Turn a finished search into a training sample + a move: the target
        is the root STATE's posterior (win, draw, loss) counts (exact spike if
        the root is solver-proven); the move is Thompson-sampled from the edge
        beliefs (near-greedy belief mean after temp_threshold)."""
        s     = self.slots[i]
        state = s['state']
        root  = s['root']
        obs   = (root.obs if root.obs is not None else
                 np.asarray(state.observation_tensor(state.current_player()),
                            dtype=np.float16))    # fp16 storage (see cell 5)
        tc = state_target(root, self.conf_cap)
        action = root_pick(root, self._rng,
                           thompson=(s['move'] < self.temp_threshold))
        pidx = int(np.nonzero(root.legal == action)[0][0])
        # Sample + transient z-mix metadata (resolved at game end, cell 5).
        s['hist'].append((obs, tc, root.player,
                          _node_solved_outcome(root) is not None))
        s['actions'].append(int(action))
        # Root value swing (posterior vs prior) → high-innovation restart seed.
        if self.restart_prob > 0 and s['root_v0'] is not None:
            swing = abs(_root_value(root) - s['root_v0'])
            if swing > s['best_swing']:
                s['best_swing'] = swing
                s['best_len'] = len(s['actions'])
        # Tree reuse: keep the played move's subtree as the next root; its
        # carried-over value seeds the next move's swing baseline.
        s['root'] = root.children[pidx]
        s['root_v0'] = (_root_value(s['root']) if s['root'] is not None
                        else None)
        state.apply_action(action)
        s['move'] += 1
        s['n']     = 0

In [ ]:
# ── Multiprocessing self-play + central GPU inference server ─────────────────
# Same design as the earlier notebooks: N CPU worker processes do ALL game and
# tree work (pure Python — processes sidestep the GIL) and ship their NN
# requests to ONE server thread here, which batches requests from all workers
# into single large forward passes on the GPU. With the state-value design a
# request is a stack of child-position rows (~35 per expansion), so the
# batches are an order of magnitude larger than the per-action notebook's.
# Small-batch inference (eval games, tournaments, arena) runs on CPU replicas.
#
# Windows/Jupyter constraint: spawned processes cannot pickle notebook-cell
# functions, so the worker code is written to a real module file first.
import pathlib

_WORKER_SRC = r'''# AUTO-GENERATED by chess_thompson_value_training.ipynb — do not edit by hand.
# Self-play worker module: state-value Dirichlet-MCTS tree ops + worker loop.
# Loads chess directly (pyspiel.load_game('chess')). Exists as a real file
# because Windows multiprocessing (spawn) cannot pickle functions defined
# inside notebook cells.

# ═══════════════════════════════════════════════════════════════════════════════
# Bayesian-MCTS tree ops — standalone copy of the notebook's semantics
# (Dirichlet value/draw beliefs per edge, mixture-propagation, MCTS-Solver),
# STATE-VALUE variant: the network predicts one (win, draw, loss, confidence)
# for a position; expanding a node batch-evaluates ALL its child positions and
# builds each edge's belief from the (perspective-flipped) child evaluation.
# Workers are pure CPU: no torch import; every NN evaluation is shipped to the
# main process's GPU inference server as a batch of child-position rows.
# ═══════════════════════════════════════════════════════════════════════════════
import os
import numpy as np
import pyspiel

_GAME = pyspiel.load_game('chess')
_NUM_ACTIONS = _GAME.num_distinct_actions()

_WIN, _DRAW, _LOSS = 0, 1, 2
_FLIP_TERM = np.array([_LOSS, _DRAW, _WIN], dtype=np.int8)
ALPHA_FLOOR = 0.05

_GRID_G  = 33
_GRID_X  = (np.arange(_GRID_G) + 0.5) / _GRID_G
_GRID_V  = 2.0 * _GRID_X - 1.0
_GRID_V2 = _GRID_V ** 2
_GRID_LX  = np.log(_GRID_X)
_GRID_L1X = np.log1p(-_GRID_X)
_SPIKE   = np.eye(_GRID_G)
_SPIKE_WIN, _SPIKE_LOSS, _SPIKE_DRAW = (_SPIKE[_GRID_G - 1], _SPIKE[0],
                                        _SPIKE[_GRID_G // 2])
_VLOSS_PEN = 1.0
_MAX_FRAC  = 0.0      # 0 = pure mixture propagation; 1 = hard max


# Innovation-based confidence calibration — see the tree-ops cell for the full
# derivation. lambda = empirically-estimated variance-inflation of the net's
# claimed confidence; estimated from backup innovations, applied at leaf
# construction. No caps/clamps: a calibrated net gives lambda -> 1 exactly.
_TDOF    = 4.0
_VAR_RES = (2.0 / _GRID_G) ** 2 / 12.0

def _psi(z2):
    return (_TDOF + 1.0) * z2 / (_TDOF + z2)

_zg = np.linspace(-10.0, 10.0, 8001)          # Riemann sum on a uniform grid
_PSI_NORM = float(np.sum(_psi(_zg ** 2)      # (np.trapz was removed in numpy 2)
                         * np.exp(-0.5 * _zg ** 2) / np.sqrt(2.0 * np.pi))
                  * (_zg[1] - _zg[0]))
del _zg


class _Calib:
    __slots__ = ('s', 'n', 'd')

    def __init__(self, prior_n=50.0, halflife=2000.0):
        self.s = prior_n
        self.n = prior_n
        self.d = 0.5 ** (1.0 / halflife)

    def observe(self, e2, var_sum):
        z2 = e2 / (var_sum + 2.0 * _VAR_RES)
        self.s = self.s * self.d + _psi(z2) / _PSI_NORM
        self.n = self.n * self.d + 1.0

    def lam(self):
        return self.s / self.n


def _edge_mean_var(node, idx):
    p = node.edge[idx]
    m = float(p @ _GRID_V)
    return m, max(float(p @ _GRID_V2) - m * m, 0.0)


def _edge_innovation(node, idx, m0, v0, calib):
    # Post-backup innovation read-off (no mixture recompute); when the
    # expansion PROVED the edge, this same read-off scores the old belief
    # against ground truth (the edge is an exact spike, v1 = 0).
    m1, v1 = _edge_mean_var(node, idx)
    calib.observe((m0 - m1) ** 2, v0 + v1)


def _beta_pmf_rows(alpha, beta):
    logw = ((alpha[:, None] - 1.0) * _GRID_LX[None, :]
            + (beta[:, None] - 1.0) * _GRID_L1X[None, :])
    logw -= logw.max(axis=1, keepdims=True)
    w = np.exp(logw)
    return w / w.sum(axis=1, keepdims=True)


def _flip_pmf(pmf):
    return pmf[::-1].copy()


def _dirichlet_leaf_belief(alpha_w, alpha_d, alpha_l, lam=1.0):
    C = alpha_w + alpha_d + alpha_l
    Ed = alpha_d / C
    Vd = alpha_d * (alpha_w + alpha_l) / (C**2 * (C + 1.0))
    Eq = alpha_w / (alpha_w + alpha_l)
    Vq = alpha_w * alpha_l / ((alpha_w + alpha_l)**2 * (alpha_w + alpha_l + 1.0))
    EX, VX = 1.0 - Ed, Vd
    EY, VY = 2.0 * Eq - 1.0, 4.0 * Vq
    Ev = EX * EY
    Vv = EX**2 * VY + EY**2 * VX + VX * VY
    mu01  = (Ev + 1.0) / 2.0
    var01 = np.clip(Vv / 4.0 * lam, 1e-9, None)
    var01 = np.minimum(var01, mu01 * (1.0 - mu01) * 0.999)
    conc  = np.maximum(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    a_beta = np.maximum(mu01 * conc, ALPHA_FLOOR)
    b_beta = np.maximum((1.0 - mu01) * conc, ALPHA_FLOOR)
    return _beta_pmf_rows(a_beta, b_beta), Ed


def _prob_best(E, C):
    logG = np.log(np.clip(C - 0.5 * E, 1e-12, 1.0))
    loo  = np.exp(logG.sum(axis=0)[None, :] - logG)
    w = (E * loo).sum(axis=1)
    s = w.sum()
    return w / s if s > 0 else np.full(len(w), 1.0 / len(w))


class _TNode:
    # p3 (k, 3) outcome probabilities + conf (k,) ALREADY in the mover's
    # perspective — assembled by _make_node from child-state evaluations.
    __slots__ = ('player', 'legal', 'edge', 'draw', 'vloss', 'term', 'children',
                 'obs')

    def __init__(self, player, legal, p3, conf, lam=1.0):
        self.player   = player
        self.legal    = np.asarray(legal, dtype=np.int32)
        self.obs      = None      # fp16 observation, stashed at expansion time
        p  = np.asarray(p3,   dtype=np.float64)
        cf = np.asarray(conf, dtype=np.float64)
        aw = np.maximum(cf * p[:, 0], ALPHA_FLOOR)
        ad = np.maximum(cf * p[:, 1], ALPHA_FLOOR)
        al = np.maximum(cf * np.clip(p[:, 2], 0.0, None), ALPHA_FLOOR)
        self.edge, self.draw = _dirichlet_leaf_belief(aw, ad, al, lam)
        self.vloss    = np.zeros(len(legal), dtype=np.int32)
        self.term     = np.full(len(legal), -1, dtype=np.int8)
        self.children = [None] * len(legal)


# Proven outcomes are baked INTO the edge arrays at the moment term is set
# (_set_term is the ONLY way engine code marks a proven edge). Indexed by
# outcome code (_WIN, _DRAW, _LOSS).
_TERM_PMF  = np.stack([_SPIKE_WIN, _SPIKE_DRAW, _SPIKE_LOSS])
_TERM_DRAW_VAL = np.array([0.0, 1.0, 0.0])
_TERM_P3   = np.eye(3)


def _set_term(node, idx, outcome):
    node.term[idx] = outcome
    node.edge[idx] = _TERM_PMF[outcome]
    node.draw[idx] = _TERM_DRAW_VAL[outcome]


def _effective_edges(node):
    return node.edge          # proven rows already spiked (see _set_term)


def _effective_draw(node):
    return node.draw          # proven rows already exact (see _set_term)


def _node_beliefs(node):
    E = _effective_edges(node)
    D = _effective_draw(node)
    if E.shape[0] == 1:
        return E[0].copy(), float(D[0])
    C = np.cumsum(E, axis=1)
    np.clip(C, 0.0, 1.0, out=C)
    w = _prob_best(E, C)
    mix = w @ E
    mean_draw = float(w @ D)
    if _MAX_FRAC > 0.0:
        Fmax = C.prod(axis=0)
        mx = np.empty(_GRID_G)
        mx[0]  = Fmax[0]
        mx[1:] = np.diff(Fmax)
        np.maximum(mx, 0.0, out=mx)
        ms = mx.sum()
        mix = (1.0 - _MAX_FRAC) * mix + _MAX_FRAC * (mx / ms if ms > 0 else mix)
    s = mix.sum()
    return (mix / s if s > 0 else np.full(_GRID_G, 1.0 / _GRID_G)), mean_draw


def _sample_edge_values(node, rng):
    C = np.cumsum(_effective_edges(node), axis=1)
    u = rng.random_sample(len(node.legal))
    idx = np.minimum((C < u[:, None]).sum(axis=1), _GRID_G - 1)
    v = _GRID_V[idx].copy()
    if node.vloss.any():
        v -= _VLOSS_PEN * node.vloss
    return v


def _select_leaf(root, root_state, rng):
    # Selection can never DISCOVER a terminal here: every terminal child was
    # marked proven when its parent was expanded (see _expand_children).
    node  = root
    state = root_state.clone()
    path  = []
    while True:
        idx = int(_sample_edge_values(node, rng).argmax())
        node.vloss[idx] += 1
        path.append((node, idx))
        if node.term[idx] >= 0:
            return path, None, None
        state.apply_action(int(node.legal[idx]))
        child = node.children[idx]
        if child is None:
            return path, state, (node, idx)
        node = child


def _backup(path):
    for node, idx in reversed(path):
        node.vloss[idx] -= 1
        if node.term[idx] < 0 and node.children[idx] is not None:
            vpmf, mdraw = _node_beliefs(node.children[idx])
            node.edge[idx] = _flip_pmf(vpmf)
            node.draw[idx] = mdraw


def _expand_children(state):
    """Apply every legal action once. Terminal children yield their exact
    outcome (mover's perspective — no flip needed later); live children yield
    one fp16 observation row (child mover's perspective) for the NN batch."""
    legal = state.legal_actions()
    me    = state.current_player()
    term  = np.full(len(legal), -1, dtype=np.int8)
    rows, row_map = [], []
    for i, a in enumerate(legal):
        ch = state.clone()
        ch.apply_action(int(a))
        if ch.is_terminal():
            r = ch.returns()[me]
            term[i] = _WIN if r > 0 else (_LOSS if r < 0 else _DRAW)
        else:
            rows.append(np.asarray(ch.observation_tensor(ch.current_player()),
                                   dtype=np.float16))
            row_map.append(i)
    return legal, term, rows, np.asarray(row_map, dtype=np.int32)


def _make_node(player, legal, term, p3_rows, conf_rows, row_map, lam=1.0):
    """Assemble a _TNode from evaluated child rows: each live row is reversed
    ([::-1] swaps win<->loss — the exact Dirichlet perspective flip, the
    child's mover being this node's opponent); terminal edges get exact
    proof spikes via _set_term."""
    k  = len(legal)
    p3 = np.full((k, 3), 1.0 / 3.0, dtype=np.float64)
    cf = np.ones(k, dtype=np.float64)
    if len(row_map):
        p3[row_map] = np.asarray(p3_rows, dtype=np.float64)[:, ::-1]
        cf[row_map] = np.asarray(conf_rows, dtype=np.float64)
    node = _TNode(player, legal, p3, cf, lam)
    for i in np.nonzero(term >= 0)[0]:
        _set_term(node, int(i), int(term[i]))
    return node


def _node_solved_outcome(node):
    """Proven outcome for node.player if solved, else None (MCTS-Solver): WIN
    as soon as any edge is a proven win; else DRAW if any edge is a proven draw
    and all edges are proven; LOSS only once every edge is a proven loss."""
    t = node.term
    if (t == _WIN).any():
        return _WIN
    if (t >= 0).all():
        return _DRAW if (t == _DRAW).any() else _LOSS
    return None


def _solver_sweep(path, aux=None, conf_cap=None):
    # Walk the path deepest-edge-first; prove each edge whose child has become
    # solved (flipped), emitting each NEWLY-solved node once as a
    # solver-labeled (obs, counts) sample into `aux` — see the tree-ops cell.
    for k in range(len(path) - 1, -1, -1):
        parent, idx = path[k]
        child = parent.children[idx]
        if child is None:
            break
        out = _node_solved_outcome(child)
        if out is None or parent.term[idx] >= 0:
            break
        _set_term(parent, idx, int(_FLIP_TERM[out]))
        if aux is not None and child.obs is not None:
            aux.append((child.obs, state_target(child, conf_cap)))


def state_target(node, conf_cap):
    # Node posterior -> DENSE (3,) fp32 state-value training counts; exact
    # spike for solver-proven nodes; concentration capped at conf_cap.
    out = _node_solved_outcome(node)
    if out is not None:
        cnt = np.full(3, ALPHA_FLOOR, dtype=np.float32)
        cnt[out] = conf_cap - 2.0 * ALPHA_FLOOR   # total exactly conf_cap
        return cnt
    vpmf, d = _node_beliefs(node)
    mean_v = float(vpmf @ _GRID_V)
    var_v  = max(float(vpmf @ _GRID_V2) - mean_v * mean_v, 1e-9)
    mu01   = (mean_v + 1.0) / 2.0
    var01  = max(var_v / 4.0, 1e-9)
    conc   = max(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    conc   = min(conc, conf_cap)
    nondraw = min(max(1.0 - d, 0.0), 1.0)
    p_win  = min(max((mean_v + nondraw) / 2.0, 0.0), nondraw)
    p_loss = nondraw - p_win
    return (np.array([p_win, d, p_loss]) * conc).astype(np.float32)


def _descend(root, action):
    # Tree reuse: the played action's subtree becomes the next search's root.
    if root is None:
        return None
    hit = np.nonzero(root.legal == action)[0]
    return root.children[int(hit[0])] if len(hit) else None


def z_mix_episode(samples, returns, z_mix, z_gamma=1.0):
    # Blend the observed outcome into each sample's STATE target (proportion-
    # average, concentration preserved), RAMPED by distance from the game end:
    # w = z_mix * z_gamma^(N-1-i). No played-action caveat: the outcome is one
    # real observation of THIS position's outcome under the current policy.
    # Only for games that actually FINISHED; solver-proven targets untouched.
    out = []
    n = len(samples)
    for i, (obs, tc, player, proven) in enumerate(samples):
        w = z_mix * (z_gamma ** (n - 1 - i))
        if w > 0.0 and not proven:
            r = returns[player]
            col = _WIN if r > 0 else (_LOSS if r < 0 else _DRAW)
            tc = tc.copy()
            conc = float(tc.sum())
            tc *= (1.0 - w)
            tc[col] += w * conc
        out.append((obs, tc))
    return out


def strip_episode_meta(samples):
    # Ply-capped games: drop metadata WITHOUT mixing (no outcome observed).
    return [(o, tc) for o, tc, _p, _v in samples]


def restart_prefix(seq, rng, k_min, k_max):
    # Prefix of a stored action sequence to replay for a backward-curriculum
    # restart — drop a random k in [k_min, k_max] tail plies. None if too short.
    if len(seq) <= k_min:
        return None
    k = int(rng.randint(k_min, min(k_max, len(seq) - 1) + 1))
    return seq[:len(seq) - k]


def _root_value(node):
    vpmf, _ = _node_beliefs(node)
    return float(vpmf @ _GRID_V)


def root_pick(root, rng, thompson):
    if thompson:
        v = _sample_edge_values(root, rng)
    else:
        v = _effective_edges(root) @ _GRID_V
    return int(root.legal[int(v.argmax())])


def run_worker(worker_id, req_q, resp_q, pool_resp_q, episode_q, cfg):
    """Pipelined self-play worker (leapfrog halves, same design as the earlier
    notebooks' workers): while one half's NN request is in flight on the
    central server, the CPU does tree work for the other half. NN requests are
    tagged 'live' (the training net) or a benchmark label (a frozen opponent-
    diversity net); the server routes them and answers 'live' on resp_q, pool
    nets on pool_resp_q.

    Wire format: a request is `(worker_id, net_id, obs (n, 1280) fp16)` — one
    row per LIVE CHILD POSITION across the half's expansion wave (no legal-
    action indices needed: the state-value head has no action axis). The
    response is `(p3 (n, 3) fp32, conf (n,) fp32)` — 16 bytes per row, so the
    response queue is bandwidth-trivial by construction (the per-action
    notebook needed server-side gathering to get here)."""
    rng  = np.random.RandomState(cfg['seed'] + worker_id * 7919)
    game = _GAME
    conf_cap  = cfg['conf_cap']
    pool_prob = cfg.get('pool_prob', 0.0)
    pool_dir  = cfg.get('checkpoint_dir')
    max_plies = cfg.get('max_selfplay_plies', 400)
    z_mix     = cfg.get('z_mix', 0.5)
    z_gamma   = cfg.get('z_gamma', 1.0)
    # Backward-restart curriculum (see the tree-ops cell). Per-worker seed pool
    # of action sequences harvested from this worker's own decisive / high-
    # innovation games; FIFO-capped.
    restart_prob  = cfg.get('restart_prob', 0.0)
    restart_kmin  = cfg.get('restart_k_min', 2)
    restart_kmax  = cfg.get('restart_k_max', 30)
    restart_cap   = cfg.get('restart_pool_cap', 128)
    swing_thresh  = cfg.get('swing_thresh', 0.6)
    rand_pool_frac = cfg.get('random_pool_frac', 0.5)
    restart_pool = []
    # One calibrator per worker, persistent across games: miscalibration is a
    # property of the (continuously-updating) net, and the decayed average
    # tracks it. Its lambda ships back with every episode for logging.
    calib = _Calib()

    def _push_seed(seq):
        if len(seq) >= 2:
            restart_pool.append(list(seq))
            if len(restart_pool) > restart_cap:
                del restart_pool[0]

    def new_game():
        sims = (cfg['fast_sims'] if rng.rand() < cfg['fast_prob']
                else cfg['full_sims'])
        state = game.new_initial_state()
        actions = []
        # Restart: replay a harvested sequence up to a few plies before its end.
        if restart_pool and rng.rand() < restart_prob:
            seq = restart_pool[rng.randint(len(restart_pool))]
            pref = restart_prefix(seq, rng, restart_kmin, restart_kmax)
            if pref:
                st = game.new_initial_state()
                ok = True
                for a in pref:
                    if st.is_terminal() or a not in st.legal_actions():
                        ok = False
                        break
                    st.apply_action(int(a))
                if ok and not st.is_terminal():
                    state, actions = st, list(pref)
        slot = {'state': state, 'hist': [], 'aux': [], 'actions': actions,
                'move': 0, 'sims': sims, 'root': None, 'n': 0, 'pool': None,
                'root_v0': None, 'best_swing': -1.0, 'best_len': 0}
        # Opponent diversity: some games have one side played by a FROZEN
        # historical checkpoint (or a uniform-RANDOM mover). Only the live
        # side's moves feed the replay buffer. The random mover guarantees
        # decisive games as soon as the net beats random — seeding the pool.
        if pool_prob > 0 and rng.rand() < pool_prob:
            try:
                labels = [f[6:-3] for f in os.listdir(pool_dir)
                          if f.startswith('bench_') and f.endswith('.pt')] \
                    if pool_dir else []
            except OSError:
                labels = []
            if not labels or rng.rand() < rand_pool_frac:
                label = 'random'          # uniform-random mover, no NN eval
            else:
                label = labels[rng.randint(len(labels))]
            slot['pool'] = {'label': label, 'side': int(rng.randint(0, 2))}
        return slot

    def finish_and_reset(i):
        """Finalize a finished-or-capped game: harvest restart seeds, resolve
        z-mix (or strip, for a cutoff), ship the episode + per-game diagnostics
        (outcome, plies, restart-pool size), start a fresh game."""
        s = slots[i]
        st = s['state']
        term = st.is_terminal()
        if term:
            ret = st.returns()
            hist2 = z_mix_episode(s['hist'], ret, z_mix, z_gamma)
            result = 'draw' if (ret[0] == 0.0 and ret[1] == 0.0) else 'decisive'
            if restart_prob > 0 and result == 'decisive':
                _push_seed(s['actions'])            # decisive game
        else:
            hist2 = strip_episode_meta(s['hist'])
            result = 'cutoff'                       # hit MAX_SELFPLAY_PLIES
        # High-innovation seed: the prefix up to the biggest root value swing.
        if restart_prob > 0 and s['best_swing'] >= swing_thresh:
            _push_seed(s['actions'][:s['best_len']])
        episode_q.put((hist2 + s['aux'], calib.lam(), len(s['aux']),
                       result, int(s['move']), len(restart_pool)))
        slots[i] = new_game()

    slots = [new_game() for _ in range(cfg['games_per_worker'])]
    mid = max(1, cfg['games_per_worker'] // 2)
    halves = [list(range(mid)), list(range(mid, cfg['games_per_worker']))]

    def collect(idxs):
        """Gather root expansions + one leaf-expansion wave per game. Proven-
        edge selections are backed up immediately; pool-opponent turns and
        solver-proven roots are skipped. Returns (jobs, paths, rows): `jobs`
        are the unique expansions awaiting NN rows (each contributing its live
        children's observation rows, in order), `paths` every pending
        selection awaiting a backup."""
        jobs, paths, rows = [], [], []
        seen = set()
        for i in idxs:
            s = slots[i]
            st0 = s['state']
            pool = s['pool']
            if pool is not None and st0.current_player() == pool['side']:
                continue         # next move belongs to the frozen opponent
            if s['root'] is None:
                legal, term, rws, rmap = _expand_children(st0)
                jobs.append(('root', i, None, None, st0, legal, term, rmap,
                             len(rws)))
                rows.extend(rws)
                continue
            if _node_solved_outcome(s['root']) is not None:
                continue         # MCTS-Solver: root proven — finalized below
            wave = min(cfg['wave'], s['sims'] - s['n'])
            for _ in range(max(wave, 0)):
                path, st, edge = _select_leaf(s['root'], st0, rng)
                if st is None:                 # proven edge re-selected
                    _backup(path)
                    s['n'] += 1
                    continue
                node, idx = edge
                paths.append((i, path))
                if (id(node), idx) not in seen:
                    seen.add((id(node), idx))
                    legal, term, rws, rmap = _expand_children(st)
                    jobs.append(('leaf', i, node, idx, st, legal, term, rmap,
                                 len(rws)))
                    rows.extend(rws)
        return jobs, paths, rows

    def apply_and_advance(idxs, jobs, paths, resp):
        """Expand a completed wave (roots + leaves) from the server's response
        rows, back up every pending selection, run the solver sweeps +
        innovations, then finish any completed (or solver-proven, or
        ply-capped) searches: record the posterior state target and play the
        Thompson-sampled move."""
        if resp is not None:
            p_np, c_np = resp
        else:
            p_np = np.zeros((0, 3), dtype=np.float32)
            c_np = np.zeros(0, dtype=np.float32)
        lam, off = calib.lam(), 0
        fresh = []                    # (slot, node, idx, pre-backup mean/var)
        for kind, i, node, idx, st, legal, term, rmap, nr in jobs:
            child = _make_node(st.current_player(), legal, term,
                               p_np[off:off + nr], c_np[off:off + nr],
                               rmap, lam)
            off += nr
            child.obs = np.asarray(st.observation_tensor(st.current_player()),
                                   dtype=np.float16)
            if kind == 'root':
                slots[i]['root'] = child
                # Prior (pre-search) root value — one leg of the per-move
                # value swing used for high-innovation restart harvesting.
                slots[i]['root_v0'] = _root_value(child)
            else:
                m0, v0 = _edge_mean_var(node, idx)
                node.children[idx] = child
                fresh.append((i, node, idx, m0, v0))
        for i, path in paths:
            _backup(path)                       # mixture-propagate the wave
            slots[i]['n'] += 1
        # Solver proofs (a child born solved proves its parent edge upward),
        # then innovations read off the backed-up/proven edges.
        fresh_paths = {}
        for i, path in paths:
            fresh_paths.setdefault((id(path[-1][0]), path[-1][1]), path)
        for i, node, idx, m0, v0 in fresh:
            path = fresh_paths.get((id(node), idx))
            if path is not None:
                _solver_sweep(path, slots[i]['aux'], conf_cap)
            _edge_innovation(node, idx, m0, v0, calib)
        for i in idxs:
            s = slots[i]
            if s['root'] is None:
                continue
            if s['n'] < s['sims'] and _node_solved_outcome(s['root']) is None:
                continue
            state = s['state']
            root = s['root']
            o = (root.obs if root.obs is not None else
                 np.asarray(state.observation_tensor(state.current_player()),
                            dtype=np.float16))
            tc = state_target(root, conf_cap)
            action = root_pick(root, rng,
                               thompson=(s['move'] < cfg['temp_threshold']))
            pidx = int(np.nonzero(root.legal == action)[0][0])
            # Sample + transient z-mix metadata (resolved at game end).
            s['hist'].append((o, tc, root.player,
                              _node_solved_outcome(root) is not None))
            s['actions'].append(int(action))
            # Root value swing (posterior vs prior): the position the net was
            # most wrong about becomes a high-innovation restart seed.
            if restart_prob > 0 and s['root_v0'] is not None:
                swing = abs(_root_value(root) - s['root_v0'])
                if swing > s['best_swing']:
                    s['best_swing'] = swing
                    s['best_len'] = len(s['actions'])
            # Tree reuse: keep the played move's subtree as the next root; its
            # carried-over value seeds the next move's swing baseline.
            s['root'] = root.children[pidx]
            s['root_v0'] = (_root_value(s['root']) if s['root'] is not None
                            else None)
            state.apply_action(action)
            s['move'] += 1
            s['n'] = 0
            if state.is_terminal() or s['move'] >= max_plies:
                finish_and_reset(i)

    def resolve_pool_moves(idxs):
        """For any game whose CURRENT mover is the frozen-opponent seat, resolve
        that single move. A 'random' opponent plays a uniform legal move with no
        NN eval; any other label routes ONE child-batch request to that frozen
        net (greedy flipped-value argmax; terminal children exact — the frozen
        opponent natively plays mates). These moves are never training targets.
        Turns strictly alternate, so at most one such move is pending per game."""
        for i in idxs:
            s = slots[i]
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            if pool['label'] == 'random':
                legal = state.legal_actions()
                action = int(legal[rng.randint(len(legal))])
            else:
                legal, term, rws, rmap = _expand_children(state)
                v = np.zeros(len(legal))
                if len(rmap):
                    req_q.put((worker_id, pool['label'], np.stack(rws)))
                    p3, _c = pool_resp_q.get()
                    # child rows are in the CHILD mover's perspective: the
                    # pool mover's value of row j is p_loss_child - p_win_child
                    v[rmap] = p3[:, _LOSS] - p3[:, _WIN]
                ti = np.nonzero(term >= 0)[0]
                if len(ti):                      # exact terminal outcomes
                    v[ti] = np.where(term[ti] == _WIN, 1.0,
                                     np.where(term[ti] == _LOSS, -1.0, 0.0))
                action = int(legal[int(np.argmax(v))])
            # Keep the live side's reused tree in sync with the opponent's move.
            s['root'] = _descend(s['root'], action)
            state.apply_action(action)
            s['actions'].append(action)
            s['move'] += 1
            if state.is_terminal() or s['move'] >= max_plies:
                finish_and_reset(i)

    # Leapfrog loop. Send/receive alternate A,B,A,B so the per-worker FIFO
    # response queue always delivers the response for the half we expect;
    # the `sent` flag keeps alignment when a half had nothing to evaluate.
    inflight = [None, None]          # per half: (jobs, paths, sent)
    while True:
        for h in (0, 1):
            if inflight[h] is not None:
                jobs, paths, sent = inflight[h]
                resp = resp_q.get() if sent else None
                apply_and_advance(halves[h], jobs, paths, resp)
                inflight[h] = None
            resolve_pool_moves(halves[h])
            jobs, paths, rows = collect(halves[h])
            sent = False
            if rows:
                req_q.put((worker_id, 'live', np.stack(rows)))
                sent = True
            inflight[h] = (jobs, paths, sent)
'''

pathlib.Path('chess_thompson_value_worker.py').write_text(_WORKER_SRC, encoding='utf-8')

import multiprocessing as _mp
import threading
import queue as _queue
import time as _time
import os as _os
import sys as _sys


class MPSelfPlayPool:
    """n_workers CPU processes play games; an inference-server thread in this
    process batches ALL their NN requests into single forward passes on
    `device`. Exposes the same .episodes() generator as ThompsonParallelSelfPlay.
    `lock` serializes model access between the server thread and training.

    The state-value head makes the server trivially bandwidth-efficient with
    no gathering machinery: a request is a stack of fp16 child-position rows,
    a response is (p3 (n, 3), conf (n,)) fp32 — 16 bytes per evaluated
    position on the response queue (the per-action notebook needed on-device
    index_select gathering to avoid shipping 74.8 KB dense rows; here the
    dense output IS 16 bytes)."""

    def __init__(self, network, device, n_workers, cfg, batch_window_s=0.002,
                 checkpoint_dir=None, channels=None, num_blocks=None):
        self.network = network
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}          # benchmark label -> frozen CPU net
        self.lock = threading.Lock()
        self._stop = threading.Event()
        ctx = _mp.get_context('spawn')            # Windows-compatible
        self.req_q = ctx.Queue()
        self.episode_q = ctx.Queue(maxsize=64)    # backpressure on workers
        self.resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.pool_resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.window = batch_window_s
        # Initialize the autograd engine's per-device state from the MAIN
        # thread before any other thread touches the device: otherwise the
        # first backward races the server thread's device use and trips
        # torch-directml's "device_ready_queues_" INTERNAL ASSERT.
        if str(device) != 'cpu':
            _t = torch.zeros(4, device=device, requires_grad=True)
            (_t * 2.0).sum().backward()
        if _os.getcwd() not in _sys.path:
            _sys.path.insert(0, _os.getcwd())
        import chess_thompson_value_worker
        self.procs = [ctx.Process(target=chess_thompson_value_worker.run_worker,
                                  args=(i, self.req_q, self.resp_qs[i],
                                        self.pool_resp_qs[i],
                                        self.episode_q, cfg), daemon=True)
                      for i in range(n_workers)]
        for p in self.procs:
            p.start()
        self.server = threading.Thread(target=self._serve, daemon=True)
        self.server.start()

    def _get_net(self, net_id):
        """'live' -> the training network on its real device (needs the lock:
        DirectML thread-safety + training mutates its weights concurrently).
        Any other net_id -> a frozen benchmark checkpoint, lazily loaded once
        and cached, always run on CPU (no lock: never mutated)."""
        if net_id == 'live':
            return self.network, self.device, True
        net = self._pool_nets.get(net_id)
        if net is None:
            path = _os.path.join(self.checkpoint_dir, f'bench_{net_id}.pt')
            net = ThompsonValueChessNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[net_id] = net
        return net, 'cpu', False

    def _serve(self):
        while not self._stop.is_set():
            try:
                reqs = [self.req_q.get(timeout=0.1)]
            except _queue.Empty:
                continue
            # Short batching window: gather concurrent workers' requests so
            # the GPU sees one big batch instead of several small ones. Capped
            # by accumulated ROWS (child-position evals) — one request is a
            # whole expansion wave (~games/2 × wave × ~35 rows), so the cap is
            # 4× the per-action notebook's (rows are ~7× cheaper through the
            # smaller net; big batches are the whole point of this design).
            rows = reqs[0][2].shape[0]
            deadline = _time.monotonic() + self.window
            while _time.monotonic() < deadline and rows < 4096:
                try:
                    r = self.req_q.get_nowait()
                    reqs.append(r)
                    rows += r[2].shape[0]
                except _queue.Empty:
                    _time.sleep(0.0003)
            # Group by net_id: almost always all 'live', occasionally a request
            # or two tagged with a frozen benchmark label (opponent-diversity
            # games) mixed in.
            groups = {}
            for wid, net_id, obs in reqs:
                groups.setdefault(net_id, []).append((wid, obs))
            for net_id, group in groups.items():
                obs = np.concatenate([o for _, o in group], axis=0)
                xin = obs.reshape(-1, *_OBS_SHAPE).astype(np.float32)  # fp16 wire
                x = torch.from_numpy(xin)
                # EVERY device op on the LIVE net under the lock (upload,
                # forward, download): DirectML is not safe under concurrent
                # access from two threads. CPU pool nets: no lock. The head
                # output is (n,3)+(n,) — the full download is 16 bytes/row,
                # no gathering needed. A failed batch (device hiccup, missing
                # benchmark file) must NOT kill this thread — every waiting
                # worker would deadlock on its response queue — so answer
                # with a NEUTRAL prior (uniform outcome, minimum confidence:
                # search evidence immediately dominates it) and keep serving,
                # loudly.
                try:
                    net, net_device, needs_lock = self._get_net(net_id)
                    if needs_lock:
                        with self.lock, torch.no_grad():
                            p, c = net(x.to(net_device))
                            p_np, c_np = p.cpu().numpy(), c.cpu().numpy()
                    else:
                        with torch.no_grad():
                            p, c = net(x)
                            p_np, c_np = p.numpy(), c.numpy()
                except Exception:
                    import traceback
                    traceback.print_exc()
                    print(f"inference server: batch for '{net_id}' failed — "
                          f"answering with neutral priors", flush=True)
                    p_np = np.full((xin.shape[0], 3), 1.0 / 3.0, np.float32)
                    c_np = np.full(xin.shape[0], CONF_MIN, np.float32)
                # Respond per request, rows in request order.
                target_qs = self.resp_qs if net_id == 'live' else self.pool_resp_qs
                off = 0
                for wid, o in group:
                    n = o.shape[0]
                    target_qs[wid].put((p_np[off:off + n], c_np[off:off + n]))
                    off += n

    def episodes(self):
        # Workers ship (samples, lambda, n_aux, result, plies, restart_pool).
        # The samples already include the solver-labeled auxiliary ones; the
        # rest are per-game diagnostics accumulated into cumulative counters
        # (self.stats) the training loop diffs each eval window.
        self.last_lam = 1.0
        self.last_aux = 0
        self.last_restart_pool = 0
        self.stats = {'games': 0, 'draw': 0, 'cutoff': 0, 'decisive': 0,
                      'plies': 0}
        while True:
            samples, lam, n_aux, result, plies, rpool = self.episode_q.get()
            self.last_lam = lam
            self.last_aux = n_aux
            self.last_restart_pool = rpool
            self.stats['games'] += 1
            self.stats[result] = self.stats.get(result, 0) + 1
            self.stats['plies'] += plies
            yield samples

    def shutdown(self):
        self._stop.set()
        try:
            self.server.join(timeout=2.0)
        except Exception:
            pass
        for p in self.procs:
            p.terminate()
        for p in self.procs:
            p.join(timeout=2.0)
        for q in ([self.req_q, self.episode_q] + self.resp_qs
                  + self.pool_resp_qs):
            try:
                q.close(); q.cancel_join_thread()
            except Exception:
                pass

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# Ported from the per-action chess Thompson notebook where the knob is
# algorithm-level; the SIM BUDGETS and NETWORK SIZE are re-derived for the
# state-value design:
#
#   Cost model: expanding a node now evaluates ALL its live children (~35
#   rows) through the net instead of 1 row, but the net is ~7× cheaper per
#   row (64ch×6 blocks ≈ 0.6M params vs 128ch×10 + two 4674-way heads ≈
#   12.7M — and the conv body, not the head, dominated forward FLOPs). Net
#   GPU work per simulation is ~5× the old design's, so the budgets below
#   are ~4× smaller — leaving total GPU work per move roughly equal while
#   arriving in ~35×-bigger batches (the regime DirectML actually likes).
#   Information per simulation is far HIGHER: a 64-sim tree holds ~64
#   expanded nodes = ~2200 real position evaluations (the old 250-sim tree
#   held 250), every edge grounded in an evaluation of its resulting
#   position, and every mate/stalemate one ply below any expanded node is
#   proven at expansion (mate-in-1 is proven before the first simulation).
NUM_EPISODES     = 50_000  # run budget only — the LR schedule no longer
                           # depends on it, so extend freely
FULL_MCTS_SIMS   = 256     # high-quality self-play (25% of games)
FAST_MCTS_SIMS   = 64      # fast self-play (75%) — each sim expands one node
                           # = one batched eval of ~35 child positions
FAST_GAME_PROB   = 0.75    # fraction of episodes using FAST_MCTS_SIMS
MAX_SELFPLAY_PLIES = 400   # self-play pacing safety valve only — SAFE to cut
                           # off: a cut-off game's samples keep their pure
                           # search targets (Z_MIX below applies only to games
                           # that actually FINISH), so nothing is invalidated.
# ── Hardware parallelism (AMD GPU via DirectML) ──────────────────────────────
# Each expansion ships ~35 rows, so per-worker requests are already large
# (games/2 × wave × ~35 ≈ 1100 rows at the defaults below) — fewer concurrent
# games per worker are needed than in the per-action notebook to keep the GPU
# fed, and the extra CPU work per expansion (one clone + observation_tensor
# per child) makes leaner workers the right trade.
import os as _os_hw
USE_WORKERS      = True    # multiprocessing self-play with a central GPU server
SELFPLAY_WORKERS = min(8, max(2, (_os_hw.cpu_count() or 8) - 2))
GAMES_PER_WORKER = 16      # per worker, split into two pipelined halves of 8
WORKER_WAVE      = 4       # expansions/game/wave (~35 NN rows each)
N_PARALLEL_GAMES = 8       # (single-process fallback when USE_WORKERS=False)
WAVE_PER_GAME    = 4       # MCTS expansions per game per wave
TEMP_THRESHOLD   = 20      # Thompson-sampled moves per game; posterior-mean
                           # argmax afterwards (the annealing analogue)
CONF_CAP         = 100.0   # posterior targets rescaled (mean-preserving) to at
                           # most this total count — caps certainty growth
                           # across generations (CONF_MAX clamps the net's side)
Z_MIX            = 0.5     # observed-outcome blend: each finished game's final
                           # result is one REAL observation of every recorded
                           # position's outcome under the current policy — the
                           # state target becomes (1-w)*search + w*onehot(z),
                           # concentration preserved (see z_mix_episode in
                           # cell 5: no played-action caveat for state targets;
                           # skipped for ply-capped games and solver-proven
                           # targets). 0 restores pure-search targets.
Z_GAMMA          = 0.97    # end-of-game ramp on the z-mix weight:
                           # w = Z_MIX * Z_GAMMA^(plies-from-end). z is nearly
                           # noiseless near the end, mostly noise far from it.
# ── Backward-restart curriculum (see cell 5) ─────────────────────────────────
# Restart a fraction of games from a few plies before endings HARVESTED FROM
# THE NET'S OWN decisive / high-innovation games (action-replay, so repetition
# semantics are exact): short, truth-dense games that feed MCTS-Solver, tree
# reuse, and the ramped z-mix. No outside knowledge; sound because no target
# depends on where a game started.
RESTART_PROB     = 0.30    # fraction of new games seeded from the restart pool
RESTART_K_MIN    = 2       # min tail plies dropped when replaying a seed
RESTART_K_MAX    = 30      # max tail plies dropped (one seed → many positions)
RESTART_POOL_CAP = 128     # per-worker FIFO cap on stored seed sequences
SWING_THRESH     = 0.6     # |posterior−prior| root value swing (of 2.0 max) to
                           # harvest a move as a high-innovation restart seed
RANDOM_POOL_FRAC = 0.5     # among opponent-pool games, fraction played vs a
                           # uniform-RANDOM mover (guarantees decisive games as
                           # soon as the net beats random — seeds the pool)
SELFPLAY_POOL_PROB = 0.15  # fraction of self-play games where one random side
                           # is played by a FROZEN benchmark instead of the live
                           # net mirror-matching itself (only the live side's
                           # moves are training targets). 0 disables it.
EVAL_EVERY       = 50      # QUICK eval: live net vs the most recent benchmark
DEEP_EVAL_EVERY  = 500     # DEEP eval: round-robin tournaments + new benchmark
QUICK_EVAL_GAMES = 20      # games vs the most recent benchmark (quick pulse)
TOURNEY_GAMES_PER_PAIR = 20  # round-robin games per pair (10 each colour)
TOURNEY_FULL_RR  = False   # linear-cost tournaments (new model vs everyone +
                           # REFRESH_PAIRS random old pairs)
REFRESH_PAIRS    = 4
# Independent evaluation tracks: (name, MCTS simulations per move). Each track
# keeps its OWN Elo table and W/D/L matrix. sims=0 is the search-free 1-ply
# value player (all child positions evaluated, argmax — the state-value
# analogue of the old raw-policy track; it natively plays mates-in-1).
EVAL_TRACKS      = [('value1', 0), ('mcts16', 16), ('mcts64', 64)]
EVAL_OPENING_PLIES = 4     # random opening half-moves per eval game (variety)
START_ELO        = 1000.0  # random and the initial net seed here
K_FACTOR         = 32.0    # Elo update step
BATCH_SIZE       = 512
TRAIN_STEPS_PER_EP = 8     # gradient steps per self-play game
# Buffer sized on the fresher side deliberately (see the per-action notebook's
# discussion of stale-buffer confidence feedback): ~100k positions / ~60
# plies per game ≈ 1600 most-recent games of memory. Samples are ~2.6 KB
# (fp16 obs + a 3-float target) → ~260 MB.
MAX_BUFFER       = 100_000
CHANNELS         = 64      # SMALLER than the per-action net's 128×10: every
NUM_BLOCKS       = 6       # parameter now works on ONE value estimate — no
                           # capacity is spent ranking 4674 head slots. ~0.6M
                           # params (was 12.7M), ~7× cheaper per forward row.
LR_PEAK          = 2e-3
LR_WARMUP_EPS    = 100      # linear warmup episodes
LR_DECAY_EPS     = 30_000   # cosine horizon — decoupled from NUM_EPISODES
WEIGHT_DECAY     = 1e-4
LR_MIN_FACTOR    = 0.10     # cosine decay floor (fraction of LR_PEAK)
CHECKPOINT_DIR   = 'chess_checkpoints_thompson_value'
RESUME_FROM_CHECKPOINT = True  # auto-resume from CHECKPOINT_DIR/latest.pt

# ── Setup ─────────────────────────────────────────────────────────────────────
game         = GAME                                           # from cell 2
base_network = ThompsonValueChessNet(CHANNELS, NUM_BLOCKS).to(DEVICE)
network      = base_network                                   # shares weights w/ compiled

# torch.compile: same policy as the earlier notebooks — skipped on DirectML and
# whenever Inductor has no C++ backend; falls back to eager on lazy failure.
network = base_network
if _BACKEND != 'directml' and hasattr(torch, 'compile'):
    import shutil, platform, logging
    _cc = (shutil.which('cl') if platform.system() == 'Windows'
           else shutil.which('g++') or shutil.which('gcc') or shutil.which('clang'))
    if _cc is None:
        print('torch.compile: skipped (no C++ compiler for Inductor) — eager mode')
    else:
        logging.getLogger('torch.fx.experimental.symbolic_shapes').setLevel(logging.ERROR)
        try:
            _compiled = torch.compile(base_network, dynamic=True)
            with torch.no_grad():                   # force compilation now
                _compiled(torch.zeros(1, *_OBS_SHAPE, device=DEVICE))
            network = _compiled
            print('torch.compile: enabled')
        except Exception as _e:
            print(f'torch.compile: disabled ({type(_e).__name__}) — running eager mode')

class LerpFreeAdamW(torch.optim.Optimizer):
    """AdamW built from mul_/add_/addcmul_/addcdiv_ only — DirectML lacks
    aten::lerp, which torch's AdamW uses in both its foreach and single-tensor
    paths (each step would round-trip every parameter through the CPU).
    Identical math to torch.optim.AdamW (decoupled decay, bias correction)."""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,
                                      weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2) = group['lr'], group['betas']
            eps, wd = group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                st = self.state[p]
                if not st:
                    st['step'] = 0
                    st['exp_avg'] = torch.zeros_like(p)
                    st['exp_avg_sq'] = torch.zeros_like(p)
                st['step'] += 1
                t = int(st['step'])
                m, v = st['exp_avg'], st['exp_avg_sq']
                p.mul_(1.0 - lr * wd)                     # decoupled decay
                m.mul_(b1).add_(p.grad, alpha=1.0 - b1)   # lerp-free
                v.mul_(b2).addcmul_(p.grad, p.grad, value=1.0 - b2)
                bc1 = 1.0 - b1 ** t
                bc2 = 1.0 - b2 ** t
                denom = (v.sqrt() / (bc2 ** 0.5)).add_(eps)
                p.addcdiv_(m, denom, value=-(lr / bc1))
        return loss


if _BACKEND == 'directml':
    optimizer = LerpFreeAdamW(network.parameters(),
                              lr=LR_PEAK, weight_decay=WEIGHT_DECAY)
else:
    optimizer = torch.optim.AdamW(network.parameters(),
                                  lr=LR_PEAK, weight_decay=WEIGHT_DECAY)

def _lr_lambda(ep):
    if ep < LR_WARMUP_EPS:
        return ep / max(LR_WARMUP_EPS, 1)
    frac = min(1.0, (ep - LR_WARMUP_EPS) / max(LR_DECAY_EPS - LR_WARMUP_EPS, 1))
    # float(): np.cos returns a numpy scalar, which otherwise lands in the
    # scheduler state and trips torch.load(weights_only=True) on resume.
    return float(max(LR_MIN_FACTOR, 0.5 * (1.0 + np.cos(np.pi * frac))))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)


# ── Loss ──────────────────────────────────────────────────────────────────────
# Targets are DENSE per-position evidence counts: (B, 3) fp32 — one
# (win, draw, loss) pseudo-count vector per sample, nothing sparse to gather
# (the whole head output is (B,3)+(B,) — smaller than one row of the old
# per-action head). Two losses are available (LOSS_FN), identical formulas to
# the per-action notebook, now applied per-position instead of per-edge:
#
#   'kl'       KL( Dirichlet(evidence) ‖ Dirichlet(predicted prior) ). Matches
#              the whole distribution, including concentration — the
#              self-referential path that lets confidence inflate unearned.
#
#   'evidence' Negative Dirichlet-Multinomial marginal log-likelihood of the
#              evidence counts under the predicted prior. The prior is SCORED
#              against the evidence, never matched to a target containing
#              itself — confidence has to be earned.

def dirichlet_kl_loss(p_sel, c_sel, t_sel):
    """KL( Dirichlet(target counts) ‖ Dirichlet(conf·probs) ), averaged over
    the batch. p_sel: (B,3); c_sel: (B,); t_sel: (B,3)."""
    t    = t_sel.clamp_min(ALPHA_FLOOR)
    b    = (c_sel.unsqueeze(-1) * p_sel).clamp_min(ALPHA_FLOOR)
    t0   = t.sum(dim=-1)
    b0   = b.sum(dim=-1)
    kl = (torch.lgamma(t0) - torch.lgamma(t).sum(dim=-1)
          - torch.lgamma(b0) + torch.lgamma(b).sum(dim=-1)
          + ((t - b) * (torch.digamma(t)
                        - torch.digamma(t0).unsqueeze(-1))).sum(dim=-1))
    return kl.mean()


def dirichlet_multinomial_loss(p_sel, c_sel, t_sel):
    """Negative Dirichlet-Multinomial marginal log-likelihood of the evidence
    counts `t_sel` (win, draw, loss) under the predicted prior
    Dirichlet(A), A = conf·probs:  −[logB(A+n) − logB(A)], with
    logB(x) = sum_k lgamma(x_k) − lgamma(sum_k x_k). Averaged over the batch.
    Non-self-referential: confidence is rewarded only when the predicted mean
    is right."""
    n = t_sel
    A = (c_sel.unsqueeze(-1) * p_sel).clamp_min(ALPHA_FLOOR)
    log_ev = ((torch.lgamma(A + n).sum(dim=-1) - torch.lgamma((A + n).sum(dim=-1)))
              - (torch.lgamma(A).sum(dim=-1) - torch.lgamma(A.sum(dim=-1))))
    return -log_ev.mean()


LOSS_FN = 'evidence'          # 'evidence' (Dirichlet-Multinomial) | 'kl' (for A/B tests)

def _search_loss(p_sel, c_sel, t_sel):
    return (dirichlet_multinomial_loss if LOSS_FN == 'evidence'
            else dirichlet_kl_loss)(p_sel, c_sel, t_sel)


# DirectML has no lgamma/digamma kernels (as with lerp / native_group_norm
# elsewhere in this notebook). On that backend the backward is SPLIT at the
# device boundary: the head outputs are copied to CPU, the loss and its
# gradient are computed there, and those gradients are handed back to the GPU
# graph as seed gradients. With the state-value head this is nearly free —
# the outputs are (B,3)+(B,), 16 bytes per sample, so no gathering machinery
# is needed at all (the per-action notebook had to probe index_select and
# gather ~0.5% of a (B,4674,3) tensor to make this affordable).
_KL_SPLIT = (_BACKEND == 'directml')

def loss_backward(probs, conf, t_sel):
    """Search loss + backward into the network params. `probs` (B,3) /
    `conf` (B,) are live on the training device with grad history; `t_sel`
    (B,3) CPU holds the target counts. Returns (loss value, p_sel, c_sel) —
    detached CPU copies of the predictions, reused by the training-loop
    diagnostics. `optimizer.zero_grad()` must precede this;
    `clip_grad_norm_` + `optimizer.step()` follow it."""
    dev = probs.device
    if _KL_SPLIT:
        p_cpu = probs.detach().cpu().requires_grad_(True)
        c_cpu = conf.detach().cpu().requires_grad_(True)
        loss_cpu = _search_loss(p_cpu, c_cpu, t_sel)
        loss_cpu.backward()
        torch.autograd.backward([probs, conf],
                                [p_cpu.grad.to(dev), c_cpu.grad.to(dev)])
        return float(loss_cpu.item()), p_cpu.detach(), c_cpu.detach()
    ts = t_sel.to(dev)
    loss = _search_loss(probs, conf, ts)
    loss.backward()
    return float(loss.item()), probs.detach().cpu(), conf.detach().cpu()


# Self-play source: worker pool (multiprocess + GPU server) or in-process.
import os
import threading
try:
    self_play.shutdown()               # re-running this cell: stop old workers
except (NameError, AttributeError):
    pass
if USE_WORKERS:
    _wcfg = dict(seed=int(np.random.randint(1_000_000)),
                 games_per_worker=GAMES_PER_WORKER, wave=WORKER_WAVE,
                 fast_sims=FAST_MCTS_SIMS, full_sims=FULL_MCTS_SIMS,
                 fast_prob=FAST_GAME_PROB, temp_threshold=TEMP_THRESHOLD,
                 conf_cap=CONF_CAP, pool_prob=SELFPLAY_POOL_PROB,
                 checkpoint_dir=CHECKPOINT_DIR,
                 max_selfplay_plies=MAX_SELFPLAY_PLIES, z_mix=Z_MIX,
                 z_gamma=Z_GAMMA, restart_prob=RESTART_PROB,
                 restart_k_min=RESTART_K_MIN, restart_k_max=RESTART_K_MAX,
                 restart_pool_cap=RESTART_POOL_CAP, swing_thresh=SWING_THRESH,
                 random_pool_frac=RANDOM_POOL_FRAC)
    self_play = MPSelfPlayPool(network, DEVICE, SELFPLAY_WORKERS, _wcfg,
                               checkpoint_dir=CHECKPOINT_DIR,
                               channels=CHANNELS, num_blocks=NUM_BLOCKS)
    torch.set_num_threads(max(2, (os.cpu_count() or 6) - SELFPLAY_WORKERS))
else:
    self_play = ThompsonParallelSelfPlay(game, network, DEVICE,
                                         n_parallel=N_PARALLEL_GAMES,
                                         wave_per_game=WAVE_PER_GAME,
                                         fast_sims=FAST_MCTS_SIMS,
                                         full_sims=FULL_MCTS_SIMS,
                                         fast_prob=FAST_GAME_PROB,
                                         temp_threshold=TEMP_THRESHOLD,
                                         conf_cap=CONF_CAP,
                                         pool_prob=SELFPLAY_POOL_PROB,
                                         checkpoint_dir=CHECKPOINT_DIR,
                                         channels=CHANNELS,
                                         num_blocks=NUM_BLOCKS,
                                         max_selfplay_plies=MAX_SELFPLAY_PLIES,
                                         z_mix=Z_MIX, z_gamma=Z_GAMMA,
                                         restart_prob=RESTART_PROB,
                                         restart_k_min=RESTART_K_MIN,
                                         restart_k_max=RESTART_K_MAX,
                                         restart_pool_cap=RESTART_POOL_CAP,
                                         swing_thresh=SWING_THRESH,
                                         random_pool_frac=RANDOM_POOL_FRAC)
episode_stream = self_play.episodes()
# Serializes model access between the inference-server thread and training.
MODEL_LOCK = getattr(self_play, 'lock', None) or threading.Lock()

# Small-batch inference (eval games, tournaments) runs on CPU replicas — the
# child-batch evals are ~35 rows, still latency-bound on DirectML.
EVAL_DEVICE = 'cpu'

def _cpu_sd(net):
    return {k: v.detach().cpu() for k, v in net.state_dict().items()}

def _cpu_clone(net):
    m = ThompsonValueChessNet(CHANNELS, NUM_BLOCKS)
    m.load_state_dict(_cpu_sd(net))
    m.eval()
    return m

def _to_cpu_tree(obj):
    if torch.is_tensor(obj):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: _to_cpu_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_cpu_tree(v) for v in obj]
    return obj

replay_buffer = []
# Tournament pool. `random` and the untrained `0` net both start at START_ELO.
_init_snap = _cpu_clone(base_network)
eval_net   = _cpu_clone(base_network)  # refreshed at every eval
benchmarks = [{'label': 'random', 'net': None},
              {'label': '0',      'net': _init_snap}]
# Per-track Elo tables and head-to-head matrices (independent between tracks).
elos = {name: {'random': START_ELO, '0': START_ELO} for name, _ in EVAL_TRACKS}
wdl  = {name: defaultdict(lambda: [0, 0, 0])         for name, _ in EVAL_TRACKS}
hist = {'ep': [], 'loss': [], 'v_err': [], 'conf_t': [], 'conf_p': [],
        'lam': [],                       # innovation-calibration factor (cell 5)
        # self-play + buffer diagnostics (per eval window)
        'draw_pct': [], 'cut_pct': [], 'dec_pct': [], 'mean_plies': [],
        'buf_total': [], 'buf_solved': [], 'restart_pool': [],
        'quick_ep': [], 'q_w': [], 'q_d': [], 'q_l': [],
        'deep_ep': [], 'elo_snap': []}   # elo_snap: list of {track: {label: elo}}
_prev_stats = {}                         # self-play counters at the last eval

# ── Checkpointing: survive crashes, resume indefinitely ──────────────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
_LATEST_CKPT = os.path.join(CHECKPOINT_DIR, 'latest.pt')

def _save_benchmark_net(label, net):
    if net is not None:
        torch.save(net.state_dict(),
                   os.path.join(CHECKPOINT_DIR, f'bench_{label}.pt'))

def save_checkpoint(ep):
    # Copy state to CPU first: DirectML (privateuseone) tensors may not pickle.
    with MODEL_LOCK:
        _model_sd = _to_cpu_tree(base_network.state_dict())
        _opt_sd   = _to_cpu_tree(optimizer.state_dict())
    torch.save({'ep': ep,
                'model': _model_sd,
                'optimizer': _opt_sd,
                'scheduler': scheduler.state_dict(),
                'elos': elos,
                'wdl': {name: dict(w) for name, w in wdl.items()},
                'hist': hist,
                'bench_labels': [b['label'] for b in benchmarks]}, _LATEST_CKPT)

start_ep = 1
if RESUME_FROM_CHECKPOINT and os.path.exists(_LATEST_CKPT):
    try:
        _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=True)
    except Exception:                     # older ckpt with a numpy scalar inside
        _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=False)
    base_network.load_state_dict(_ck['model'])
    optimizer.load_state_dict(_ck['optimizer'])
    scheduler.load_state_dict(_ck['scheduler'])
    elos = _ck['elos']
    wdl  = {name: defaultdict(lambda: [0, 0, 0], w)
            for name, w in _ck['wdl'].items()}
    hist = _ck['hist']
    benchmarks = [{'label': 'random', 'net': None}]
    # Dedup: repeated resumes could accumulate the same benchmark label more
    # than once, double-weighting it in tournaments and the opponent pool.
    # dict.fromkeys keeps first-seen order, one entry per label.
    for _lbl in dict.fromkeys(_ck['bench_labels']):
        if _lbl == 'random':
            continue
        _bn = ThompsonValueChessNet(CHANNELS, NUM_BLOCKS)  # benchmarks live on CPU
        _bn.load_state_dict(torch.load(
            os.path.join(CHECKPOINT_DIR, f'bench_{_lbl}.pt'),
            map_location='cpu', weights_only=True))
        _bn.eval()
        benchmarks.append({'label': _lbl, 'net': _bn})
    start_ep = _ck['ep'] + 1
    print(f'Resumed from checkpoint: episode {_ck["ep"]}, pool {len(benchmarks)}')


def _run_all_tracks(focus=None):
    for name, sims in EVAL_TRACKS:
        run_tournament(benchmarks, elos[name], wdl[name], game, EVAL_DEVICE,
                       TOURNEY_GAMES_PER_PAIR, K_FACTOR, sims,
                       opening_plies=EVAL_OPENING_PLIES,
                       focus_label=focus, refresh_pairs=REFRESH_PAIRS)
    hist['deep_ep'].append(ep_now[0])
    hist['elo_snap'].append({name: dict(elos[name]) for name, _ in EVAL_TRACKS})


def _print_tracks(prefix):
    for name, _ in EVAL_TRACKS:
        ladder = '  '.join(f'{b["label"]}={elos[name][b["label"]]:.0f}'
                           for b in sorted(benchmarks, key=lambda b: -elos[name][b['label']]))
        print(f'{prefix} {name:>7} Elos (strong→weak): {ladder}')


n_params = sum(p.numel() for p in network.parameters() if p.requires_grad)
print(f'Device: {DEVICE}  |  ThompsonValueChessNet params: {n_params:,}')
print(f'{NUM_EPISODES} eps | fast {FAST_MCTS_SIMS} sims ({FAST_GAME_PROB:.0%}) '
      f'/ full {FULL_MCTS_SIMS} sims (each sim = 1 expansion ≈ 35 child evals) '
      f'| mixture-prop + MCTS-Solver | {LOSS_FN} loss, '
      f'conf cap {CONF_CAP:.0f} | z-mix {Z_MIX:.2f}(γ{Z_GAMMA:.2f}) | '
      f'pool {SELFPLAY_POOL_PROB:.0%}(rand {RANDOM_POOL_FRAC:.0%}) | '
      f'restart {RESTART_PROB:.0%} | no aug | '
      f'{TRAIN_STEPS_PER_EP} steps/ep | ply cap {MAX_SELFPLAY_PLIES}')
print(f'Quick eval every {EVAL_EVERY} eps (vs latest benchmark); deep tournaments '
      f'every {DEEP_EVAL_EVERY} eps | tracks: {[f"{n}({s})" for n, s in EVAL_TRACKS]}')
print(f'Checkpoints: {CHECKPOINT_DIR} (resume={RESUME_FROM_CHECKPOINT})')
if USE_WORKERS:
    print(f'Self-play: {SELFPLAY_WORKERS} worker processes × {GAMES_PER_WORKER} games '
          f'(wave {WORKER_WAVE}) → GPU inference server on {DEVICE}')
print('-' * 72)

aux_total = 0        # solver-labeled auxiliary samples seen (see cell 5)
ep_now = [start_ep - 1]
if start_ep == 1:
    # Iteration-0 tournaments: random vs the untrained net, one per track.
    _save_benchmark_net('0', _init_snap)
    _run_all_tracks()
    _print_tracks('Iter     0 |')
    save_checkpoint(0)
print('-' * 72)

for ep in range(start_ep, NUM_EPISODES + 1):
    ep_now[0] = ep

    # 1. Pull the next finished self-play game. Its samples are the per-move
    #    root state targets PLUS any solver-labeled auxiliary samples
    #    (positions the MCTS-Solver proved during the game's searches — exact
    #    game-theoretic labels, same (obs, counts) format; see cell 5).
    network.eval()
    raw_data = next(episode_stream)
    _n_aux_ep = int(getattr(self_play, 'last_aux', 0))
    aux_total += _n_aux_ep

    # 2. No board-symmetry augmentation for chess (see cell 5) — augment_sample
    #    is a no-op wrapper kept only for structural parity with the earlier
    #    notebooks. Samples are (obs fp16, counts fp32 (3,), solved): ~2.6 KB
    #    each (~260 MB at MAX_BUFFER=100k). The solver-labeled auxiliary
    #    samples are the LAST _n_aux_ep of the episode — tag them so the
    #    buffer's solved-fraction is a live diagnostic.
    _n_ep = len(raw_data)
    for _k, (obs, tc) in enumerate(raw_data):
        replay_buffer.extend(
            augment_sample(obs, tc, solved=(_k >= _n_ep - _n_aux_ep)))
    if len(replay_buffer) > MAX_BUFFER:
        del replay_buffer[:-MAX_BUFFER]

    # 3. Train: the selected search loss (see LOSS_FN) on the state targets
    network.train()
    losses, v_errs, conf_ts, conf_ps = [], [], [], []
    if len(replay_buffer) >= BATCH_SIZE:
        for _ in range(TRAIN_STEPS_PER_EP):
            batch = random.sample(replay_buffer, BATCH_SIZE)
            t_sel = torch.from_numpy(
                np.stack([s[1] for s in batch]).astype(np.float32))
            # The device part of the step sits under the lock: every device op
            # must be serialized against the inference-server thread (DirectML
            # is not thread-safe).
            with MODEL_LOCK:
                x_b = batch_to_tensor([s[0] for s in batch], DEVICE)
                probs, conf = network(x_b)
                optimizer.zero_grad()
                loss_val, p_sel, c_sel = loss_backward(probs, conf, t_sel)
                torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                optimizer.step()

            with torch.no_grad():
                # Diagnostics on the CPU copies loss_backward already produced
                # — nothing extra is downloaded, no exotic op ever touches the
                # DirectML device, and this runs OUTSIDE the lock.
                t  = t_sel.clamp_min(ALPHA_FLOOR)
                t0 = t.sum(dim=-1)
                tv = (t[:, 0] - t[:, 2]) / t0              # target mean value
                pv = p_sel[:, 0] - p_sel[:, 2]             # predicted mean value
                v_errs.append(float((tv - pv).abs().mean()))
                conf_ts.append(float(t0.mean()))
                conf_ps.append(float(c_sel.mean()))
            losses.append(loss_val)
        scheduler.step()

    if ep % EVAL_EVERY != 0:
        continue

    # 4. Evaluation (CPU replica: small-batch DirectML inference is latency-bound)
    network.eval()
    with MODEL_LOCK:
        eval_net.load_state_dict(_cpu_sd(base_network))
    mloss = float(np.mean(losses))  if losses  else float('nan')
    mve   = float(np.mean(v_errs))  if v_errs  else float('nan')
    mct   = float(np.mean(conf_ts)) if conf_ts else float('nan')
    mcp   = float(np.mean(conf_ps)) if conf_ps else float('nan')
    lam_now = float(getattr(self_play, 'last_lam', float('nan')))
    # Self-play outcome mix over THIS eval window (diff the cumulative counters).
    _st  = dict(getattr(self_play, 'stats', {}))
    _dg  = _st.get('games', 0)   - _prev_stats.get('games', 0)
    _dd  = _st.get('draw', 0)    - _prev_stats.get('draw', 0)
    _dc  = _st.get('cutoff', 0)  - _prev_stats.get('cutoff', 0)
    _dp  = _st.get('plies', 0)   - _prev_stats.get('plies', 0)
    _prev_stats = _st
    draw_pct = 100.0 * _dd / _dg if _dg else float('nan')
    cut_pct  = 100.0 * _dc / _dg if _dg else float('nan')
    dec_pct  = 100.0 * (_dg - _dd - _dc) / _dg if _dg else float('nan')
    mean_plies = _dp / _dg if _dg else float('nan')
    # Buffer composition: total positions + how many are solver-labeled truth.
    buf_total  = len(replay_buffer)
    buf_solved = sum(1 for _s in replay_buffer if _s[2])
    restart_pool = int(getattr(self_play, 'last_restart_pool', 0))
    hist['ep'].append(ep); hist['loss'].append(mloss); hist['v_err'].append(mve)
    hist['conf_t'].append(mct); hist['conf_p'].append(mcp)
    hist['lam'].append(lam_now)
    hist['draw_pct'].append(draw_pct); hist['cut_pct'].append(cut_pct)
    hist['dec_pct'].append(dec_pct);   hist['mean_plies'].append(mean_plies)
    hist['buf_total'].append(buf_total); hist['buf_solved'].append(buf_solved)
    hist['restart_pool'].append(restart_pool)
    _slv_pct = 100.0 * buf_solved / buf_total if buf_total else 0.0
    _diag = (f'dr={draw_pct:.0f}% cut={cut_pct:.0f}% ply={mean_plies:.0f} '
             f'buf={buf_total/1000:.0f}k/{_slv_pct:.0f}%slv rp={restart_pool}')
    lr_now = optimizer.param_groups[0]['lr']

    if ep % DEEP_EVAL_EVERY == 0:
        # DEEP: add current net as a new benchmark, warm-starting each track's
        # Elo from the previous checkpoint, then run one round-robin per track.
        new_label = str(ep)
        for name, _ in EVAL_TRACKS:
            elos[name].setdefault(new_label, elos[name][benchmarks[-1]['label']])
        with MODEL_LOCK:
            snap = _cpu_clone(base_network)
        # Idempotent under re-runs / resume-at-a-deep-eval-episode: replace the
        # net if this label already exists, never append a duplicate.
        _existing = next((b for b in benchmarks if b['label'] == new_label), None)
        if _existing is not None:
            _existing['net'] = snap
        else:
            benchmarks.append({'label': new_label, 'net': snap})
        _run_all_tracks(None if TOURNEY_FULL_RR else new_label)
        print(f'Ep {ep:5d} | loss={mloss:.3f} vErr={mve:.3f} a0 t/p={mct:.1f}/{mcp:.1f} '
              f'lam={lam_now:.2f} aux={aux_total} {_diag} '
              f'| DEEP tournaments (pool {len(benchmarks)}) '
              f'| lr={lr_now:.2e}')
        _print_tracks('        |')
        _save_benchmark_net(new_label, snap)
        save_checkpoint(ep)
    else:
        # QUICK: search-free pulse vs the most recent benchmark (no Elo change).
        ref = benchmarks[-1]
        w, d, l = play_match(eval_net, ref['net'], game,
                             QUICK_EVAL_GAMES, EVAL_DEVICE)
        hist['quick_ep'].append(ep)
        hist['q_w'].append(w); hist['q_d'].append(d); hist['q_l'].append(l)
        _t0name = EVAL_TRACKS[0][0]
        print(f'Ep {ep:5d} | loss={mloss:.3f} vErr={mve:.3f} a0 t/p={mct:.1f}/{mcp:.1f} '
              f'lam={lam_now:.2f} aux={aux_total} {_diag} '
              f'| vs {ref["label"]:>5} W{w:2d}D{d:2d}L{l:2d} '
              f'({_t0name} Elo={elos[_t0name][ref["label"]]:.0f}) | lr={lr_now:.2e}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(22, 9))
fig.suptitle('Chess ThompsonZero (state-value) Training', fontsize=14)

ax = axes[0, 0]
ax.plot(hist['ep'], hist['loss'], label=f'{LOSS_FN} loss')
ax.set_title('Search loss (falls for KL; Dirichlet-Multinomial NLL is a different scale)')
ax.set_xlabel('Episode')
ax2 = ax.twinx()
ax2.plot(hist['ep'], hist['conf_t'], color='tab:purple', alpha=0.6, label='α₀ target')
ax2.plot(hist['ep'], hist['conf_p'], color='tab:brown', alpha=0.6, label='α₀ predicted')
ax2.set_ylabel('mean α₀')
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8)

axes[0, 1].plot(hist['ep'], hist['v_err'], color='tab:orange', label='|v err|')
axes[0, 1].set_title('|value target − predicted|  +  calibration λ')
axes[0, 1].set_xlabel('Episode')
if any(np.isfinite(hist.get('lam', []))):
    axl = axes[0, 1].twinx()
    axl.plot(hist['ep'], hist['lam'], color='tab:blue', alpha=0.6, label='λ (calib)')
    axl.axhline(1.0, color='tab:blue', linestyle=':', linewidth=0.8, alpha=0.5)
    axl.set_ylabel('λ  (1 = calibrated)')
    h1, l1 = axes[0, 1].get_legend_handles_labels()
    h2, l2 = axl.get_legend_handles_labels()
    axes[0, 1].legend(h1 + h2, l1 + l2, fontsize=8)

# Elo of the model trained-to-episode-N, one line per evaluation track.
def _newest_label(ep):
    return '0' if ep == 0 else str(ep)
for name, _ in EVAL_TRACKS:
    xs, ys = [], []
    for ep, snap in zip(hist['deep_ep'], hist['elo_snap']):
        lbl = _newest_label(ep)
        if lbl in snap[name]:
            xs.append(ep); ys.append(snap[name][lbl])
    axes[0, 2].plot(xs, ys, marker='.', label=name)
axes[0, 2].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[0, 2].set_title('Current-model Elo by track')
axes[0, 2].set_xlabel('Episode'); axes[0, 2].legend(fontsize=8)

# Quick-eval progress pulse: search-free W/D/L vs the most recent benchmark.
axes[1, 0].plot(hist['quick_ep'], hist['q_w'], color='tab:green', marker='.', label='Win')
axes[1, 0].plot(hist['quick_ep'], hist['q_d'], color='gray', linestyle='--', label='Draw')
axes[1, 0].plot(hist['quick_ep'], hist['q_l'], color='tab:red', linestyle=':', label='Loss')
axes[1, 0].set_title(f'Quick eval vs latest benchmark ({QUICK_EVAL_GAMES} games, search-free)')
axes[1, 0].set_xlabel('Episode'); axes[1, 0].legend(fontsize=8)

# Final Elo per benchmark, grouped bars by track.
labels = [b['label'] for b in benchmarks]
x = np.arange(len(labels)); width = 0.8 / max(len(EVAL_TRACKS), 1)
for k, (name, _) in enumerate(EVAL_TRACKS):
    vals = [elos[name].get(l, np.nan) for l in labels]
    axes[1, 1].bar(x + (k - (len(EVAL_TRACKS) - 1) / 2) * width, vals, width, label=name)
axes[1, 1].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 1].set_title('Final Elo per benchmark'); axes[1, 1].set_ylabel('Elo')
axes[1, 1].legend(fontsize=8)

# Head-to-head win-rate matrix for the strongest track (highest sims).
mtrack = EVAL_TRACKS[-1][0]
W = wdl[mtrack]
n = len(labels)
M = np.full((n, n), np.nan)
for i, ri in enumerate(labels):
    for j, cj in enumerate(labels):
        if (ri, cj) in W:
            w, d, l = W[(ri, cj)]; tot = w + d + l
            if tot: M[i, j] = (w + 0.5 * d) / tot
        elif (cj, ri) in W:
            w, d, l = W[(cj, ri)]; tot = w + d + l
            if tot: M[i, j] = 1.0 - (w + 0.5 * d) / tot
im = axes[1, 2].imshow(M, cmap='RdYlGn', vmin=0, vmax=1)
axes[1, 2].set_xticks(range(n)); axes[1, 2].set_yticks(range(n))
axes[1, 2].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 2].set_yticklabels(labels, fontsize=7)
axes[1, 2].set_title(f'Win-rate matrix — {mtrack} (row vs col)')
for i in range(n):
    for j in range(n):
        if not np.isnan(M[i, j]):
            axes[1, 2].text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                            fontsize=6, color='black')
fig.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

# Self-play outcome mix (per eval window): the draw-density signal the whole
# curriculum targets — decisive% should rise, draw%/cutoff% fall.
ax = axes[0, 3]
if hist.get('draw_pct'):
    ax.plot(hist['ep'], hist['dec_pct'], color='tab:green', label='decisive %')
    ax.plot(hist['ep'], hist['draw_pct'], color='gray', label='draw %')
    ax.plot(hist['ep'], hist['cut_pct'], color='tab:red', linestyle=':', label='cutoff %')
    ax.set_ylim(0, 100); ax.set_ylabel('% of self-play games')
    axp = ax.twinx()
    axp.plot(hist['ep'], hist['mean_plies'], color='tab:blue', alpha=0.4,
             label='mean plies')
    axp.set_ylabel('mean plies/game')
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = axp.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7, loc='center right')
ax.set_title('Self-play outcome mix'); ax.set_xlabel('Episode')

# Buffer composition + restart-pool size: how much of the training data is
# solver-labeled ground truth, and whether the restart pool is populating.
ax = axes[1, 3]
if hist.get('buf_total'):
    bt = np.array(hist['buf_total'], dtype=float)
    bs = np.array(hist['buf_solved'], dtype=float)
    ax.plot(hist['ep'], bt / 1000.0, color='tab:purple', label='buffer (k)')
    ax.plot(hist['ep'], bs / 1000.0, color='tab:orange', label='solver-labeled (k)')
    ax.set_ylabel('positions (thousands)')
    axr = ax.twinx()
    axr.plot(hist['ep'], hist['restart_pool'], color='tab:brown', alpha=0.4,
             label='restart pool')
    axr.set_ylabel('restart-pool seeds (last worker)')
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = axr.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7, loc='center right')
ax.set_title('Buffer composition + restart pool'); ax.set_xlabel('Episode')

plt.tight_layout()
plt.show()

## Arena: pit any two checkpoints against each other

There's no existing "KataZero-chess" notebook to compare against the way the
Boop notebooks compare ThompsonZero to KataZero, so this section is a general
**checkpoint-vs-checkpoint arena** instead: load any two `ThompsonChessNet`
state dicts (any `bench_<ep>.pt` from `CHECKPOINT_DIR`, or `latest.pt`'s
`['model']` entry) and play them off, search-free or with search, alternating
colours, with a printable move-by-move game log (chess positions are
human-inspectable via FEN/SAN, unlike Boop's board).

The training cell's own DEEP-eval tournaments already give you an ongoing Elo
ladder across generations (that *is* the primary strength signal — see the
plots cell); this section is for **ad-hoc** comparisons: two specific
checkpoints, a specific `LOSS_FN` variant against another, or a specific sim
budget.

**Plugging in an external opponent** (Stockfish via UCI, another engine, a
human): `arena()` below is specifically for two `ThompsonChessNet` checkpoints.
To face something else, write your own move loop around a `pick(state) ->
action` function for the external side (any `state.legal_actions()`-compatible
choice) — `arena()`'s body is short and is a template for that; nothing here
has been tested against an external engine, so treat this as a starting point,
not a ready integration.

In [ ]:
import os


def _load_sd(path):
    # Full training checkpoints (latest.pt) also store non-tensor objects — e.g.
    # a numpy float in the LR-scheduler state (np.cos in the LR schedule) — which
    # torch's default weights_only=True unpickler rejects. Try the safe load,
    # then fall back to a full load for these trusted, locally-produced files.
    try:
        sd = torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        sd = torch.load(path, map_location='cpu', weights_only=False)
    return sd['model'] if isinstance(sd, dict) and 'model' in sd else sd


def load_chess_net(path, channels=CHANNELS, num_blocks=NUM_BLOCKS):
    net = ThompsonValueChessNet(channels, num_blocks)
    net.load_state_dict(_load_sd(path))
    net.eval()
    return net


def arena(net_a, net_b, game, n_games=20, a_sims=0, b_sims=0,
          device='cpu', opening_plies=4, verbose=True):
    """net_a vs net_b. sims == 0 -> search-free (one-ply value argmax); sims > 0
    -> full Thompson-MCTS search (posterior-mean argmax, no root noise needed).
    Alternates colours; the first `opening_plies` moves of each game are random
    for variety. Returns (wins_a, draws, wins_b) from net_a's perspective."""
    bot_a = (make_thompson_bot(game, net_a, device, a_sims,
                               batch_size=max(1, min(a_sims, 16)))
             if a_sims > 0 else None)
    bot_b = (make_thompson_bot(game, net_b, device, b_sims,
                               batch_size=max(1, min(b_sims, 16)))
             if b_sims > 0 else None)
    w = d = l = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2                 # alternate colours
        ply = 0
        while not state.is_terminal() and ply < MAX_EVAL_PLIES:
            if ply < opening_plies:
                action = random.choice(state.legal_actions())
            elif state.current_player() == a_player:
                action = (_mcts_move(bot_a, state) if bot_a is not None
                          else policy_action(net_a, state, device))
            else:
                action = (_mcts_move(bot_b, state) if bot_b is not None
                          else policy_action(net_b, state, device))
            state.apply_action(action)
            ply += 1
        # A non-terminal state's returns() is exactly [0.0, 0.0] in chess, so a
        # ply-cap cutoff scores as a draw here with no special-casing needed.
        r = state.returns()[a_player]
        if r > 0:   w += 1
        elif r < 0: l += 1
        else:       d += 1
        if verbose and (i + 1) % 5 == 0:
            print(f'    {i + 1:3d}/{n_games}  net_a W{w} D{d} L{l}')
    return w, d, l


def play_and_show(net_a, net_b, game, device='cpu', max_plies=200):
    """One game, net_a (White) vs net_b (Black), search-free, printing each
    move in SAN and the final result — a quick sanity check you can eyeball.
    Either net may be None for a random mover (e.g. no second checkpoint yet)."""
    state = game.new_initial_state()
    moves = []
    ply = 0
    while not state.is_terminal() and ply < max_plies:
        mover = state.current_player()
        net = net_a if mover == 1 else net_b     # player 1 = White (see cell 2)
        action = (random.choice(state.legal_actions()) if net is None
                  else policy_action(net, state, device))
        moves.append(state.action_to_string(mover, action))
        state.apply_action(action)
        ply += 1
    print(' '.join(f'{i//2+1}.{m}' if i % 2 == 0 else m
                   for i, m in enumerate(moves)))
    if state.is_terminal():
        print('Result:', state.returns(), '  FEN:', state)
    else:
        print(f'(stopped at the {max_plies}-ply display cap, not terminal)')
    return state


# ── Run it ────────────────────────────────────────────────────────────────────
# By default: the LATEST checkpoint vs the UNTRAINED episode-0 net and vs
# random, using whatever this session has already trained/loaded. Point
# CKPT_A / CKPT_B at any two bench_<ep>.pt files (or another CHECKPOINT_DIR
# entirely) to compare specific generations or LOSS_FN variants.
CKPT_A       = os.path.join(CHECKPOINT_DIR, 'latest.pt')   # newest
CKPT_B       = os.path.join(CHECKPOINT_DIR, 'bench_0.pt')  # untrained baseline
ARENA_GAMES  = 20
ARENA_SIMS   = 50     # per-move simulations for the with-search match
ARENA_DEVICE = 'cpu'  # small-batch inference: CPU beats DirectML here

if not os.path.exists(CKPT_A):
    print(f'No checkpoint at {CKPT_A} yet — train at least one episode first.')
else:
    net_a = load_chess_net(CKPT_A)
    net_b = load_chess_net(CKPT_B) if os.path.exists(CKPT_B) else None
    print(f'net_a: {CKPT_A}')
    print(f'net_b: {CKPT_B if net_b is not None else "(missing — using random)"}')

    if net_b is not None:
        print(f'\nSearch-free ({ARENA_GAMES} games):')
        w, d, l = arena(net_a, net_b, game, ARENA_GAMES, 0, 0, ARENA_DEVICE)
        print(f'  net_a W{w} D{d} L{l}  '
              f'({(w + 0.5 * d) / max(w + d + l, 1):.1%} score)')

        print(f'\n{ARENA_SIMS}-sim MCTS ({ARENA_GAMES} games):')
        w, d, l = arena(net_a, net_b, game, ARENA_GAMES,
                        ARENA_SIMS, ARENA_SIMS, ARENA_DEVICE)
        print(f'  net_a W{w} D{d} L{l}  '
              f'({(w + 0.5 * d) / max(w + d + l, 1):.1%} score)')

    print('\nOne search-free game vs random, shown move-by-move:')
    play_and_show(net_a, None if net_b is None else net_b, game, ARENA_DEVICE)